# Dados abertos CAPES - Modelagem para painel do GID

In [835]:
##SEMPRE EXECUTAR ESSA CÉLULA!!!!
#Importando bibliotecas necessárias:
import os
import time
import re
import ssl
import requests
from urllib.parse import urlparse
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
import json
import numpy as np
import hashlib
from pysqlcipher3 import dbapi2 as sqlite # No linux, a instalação do pysqlcipher3 pode envolver a instalação de bibliotecas como libsqlcipher-dev no sistema
from dotenv import load_dotenv
from concurrent.futures import ProcessPoolExecutor


#Diretórios usados para armazenar os dados (baixados e processados)
dirs = {
    'download_dir': 'capes_csv_files',
    'filtered_dir': 'ufrj_data',
    'processed_dir': 'sucupira_painel',
    'discentes': 'discentes',
    'docentes': 'docentes',
    'programas': 'programas',
    'cursos':  'programas',
    'producao': 'producao',
    'producao_detalhe': 'producao',
    'producao_autor': 'producao',
    'projetos': 'projetos',
    'membros': 'projetos',
    'financiadores': 'financiadores',
    'btd': 'btd',
}

#Base directories
download_dir = dirs.get('download_dir')
filtered_dir = dirs.get('filtered_dir')
processed_dir = dirs.get('processed_dir')

## Download dos dados abertos da CAPES

In [836]:
# Configurações do download
api_url = "https://dadosabertos.capes.gov.br/api/3/action/package_search"
organization = "diretoria-de-avaliacao"
output_dir = dirs.get('download_dir', 'capes_csv_files')
timeout_seconds = 30
max_retries = 3


prefix_substring_dirs = {
    "ddi-br-capes-colsucup-": {
        "projeto-financiador": dirs.get("financiadores", "financiadores"),
        "projeto": dirs.get("projetos", "projetos"),
    }, 
    "br-capes-colsucup-": {
        "prod": dirs.get("producao", "producao"),
        "producao": dirs.get("producao", "producao"),
        "projeto": dirs.get("projetos", "projetos"),
        "membro": dirs.get("projetos", "projetos"),
        "prog": dirs.get("programas", "programas"),
        "curso": dirs.get("cursos", "cursos"),
		"discentes": dirs.get("discentes", "discentes"),
		"docente": dirs.get("docentes", "docentes"),
        "financiador": dirs.get("financiadores", "financiadores"),
    },
    "br-capes-col-": {
        "proj": dirs.get("projetos", "projetos"),
        "producao": dirs.get("producao", "producao"),
        "prod": dirs.get("producao", "producao"),
    },
	"br-colsucup-": {
		"prod": dirs.get("producao", "producao"),
	},
    "br-capes-btd-": {
        "": dirs.get("btd", "btd")
    },
} # Mapeia prefix+substring para pastas (diretórios)

# Sessão com retry
session = requests.Session()
retry = Retry(total=max_retries, backoff_factor=1)
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)

# Contexto SSL customizado (caso precise ignorar erros SSL)
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

# Criar diretório base
os.makedirs(output_dir, exist_ok=True)

def sanitize_filename(filename):
    """Remove caracteres inválidos para nomes de arquivo"""
    return re.sub(r'[\\/*?:"<>|]', "_", filename)

def get_with_retry(url):
    """Faz uma requisição HTTP com retry e fallback para SSL desabilitado ou HTTP"""
    for attempt in range(max_retries):
        try:
            print(f"Tentativa {attempt + 1} para {url}")
            try:
                response = session.get(url, timeout=timeout_seconds)
                response.raise_for_status()
                return response
            except requests.exceptions.SSLError:
                print("Falha SSL, tentando com verificação desativada...")
                response = session.get(url, timeout=timeout_seconds, verify=False)
                response.raise_for_status()
                return response
        except requests.exceptions.RequestException as e:
            print(f"Erro na tentativa {attempt + 1}: {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
    if url.startswith('https://'):
        http_url = url.replace('https://', 'http://', 1)
        print(f"Tentando fallback para HTTP: {http_url}")
        try:
            response = session.get(http_url, timeout=timeout_seconds)
            response.raise_for_status()
            return response
        except requests.exceptions.RequestException as e:
            print(f"Falha no fallback HTTP: {e}")
    return None

def get_all_csv_links_from_ckan():
    """Consulta a API CKAN e retorna todos os arquivos CSV da organização"""
    print("Consultando a API CKAN da CAPES...")
    params = {
        "fq": f"organization:{organization}",
        "rows": 1000
    }
    try:
        response = session.get(api_url, params=params, timeout=timeout_seconds)
        response.raise_for_status()
        results = response.json()["result"]["results"]
        csv_links = []
        for dataset in results:
            for resource in dataset.get("resources", []):
                if resource.get("format", "").lower() == "csv":
                    url = resource.get("url")
                    if url:
                        csv_links.append(url)
        return sorted(set(csv_links))
    except Exception as e:
        print(f"Erro ao consultar CKAN: {e}")
        return []

def detect_subfolder(filename):
    """Detecta subpasta com base nas substrings do nome do arquivo"""
    name = filename.lower()
    for prefix, substrings in prefix_substring_dirs.items():
        for substr, folder in substrings.items():
            target = prefix + substr
            if target in name:
                return folder
    return "outros"


def download_csv_files(links):
    """Baixa todos os arquivos CSV e organiza em subpastas por tipo"""
    total = len(links)
    print(f"\nIniciando download de {total} arquivos...")

    for i, url in enumerate(links, 1):
        try:
            raw_filename = url.split('/')[-1].split('?')[0]
            filename = sanitize_filename(raw_filename)
            subfolder = detect_subfolder(filename)

            subdir_path = os.path.join(output_dir, subfolder)
            os.makedirs(subdir_path, exist_ok=True)

            filepath = os.path.join(subdir_path, filename)

            if os.path.exists(filepath):
                print(f"[{i}/{total}] Já existe: {subfolder}/{filename}")
                continue

            print(f"[{i}/{total}] Baixando: {filename} para {subfolder}/")

            start_time = time.time()
            response = get_with_retry(url)
            if not response:
                print(f"Falha ao baixar {url}")
                continue

            with open(filepath, 'wb') as f:
                f.write(response.content)

            size_mb = os.path.getsize(filepath) / (1024 * 1024)
            print(f"Salvo como {subfolder}/{filename} ({size_mb:.2f} MB) em {time.time() - start_time:.2f}s")

        except Exception as e:
            print(f"Erro ao baixar {url}: {e}")

In [837]:
#Código para download dos dados abertos - Processo potencialmente demorado
print("Iniciando processo para baixar todos os CSVs da Diretoria de Avaliação...")
start_time = time.time()
try:
    csv_links = get_all_csv_links_from_ckan()
    print(f"\n{len(csv_links)} arquivos CSV encontrados:")
    for i, link in enumerate(csv_links[:10], 1):
        print(f"{i}. {link.split('/')[-1].split('?')[0]}")
    if len(csv_links) > 10:
        print(f"... mais {len(csv_links) - 10} arquivos")
    if csv_links:
        download_csv_files(csv_links)
    print(f"\nConcluído em {time.time() - start_time:.2f}s")
except KeyboardInterrupt:
    print("\nProcesso interrompido pelo usuário.")
except Exception as e:
    print(f"Erro inesperado: {e}")

Iniciando processo para baixar todos os CSVs da Diretoria de Avaliação...
Consultando a API CKAN da CAPES...

429 arquivos CSV encontrados:
1. br-capes-colsucup-producao-2013a2016-2017-11-01-tecnica-cursocdu.csv
2. br-capes-colsucup-producao-2013a2016-2017-11-01-tecnica-progrtv.csv
3. br-capes-colsucup-producao-2013a2016-2017-11-01-bibliografica-tradu.csv
4. br-capes-colsucup-producao-2013a2016-2017-11-01-artistica-visual.csv
5. br-capes-colsucup-producao-2013a2016-2017-11-01-tecnica-destec.csv
6. br-capes-colsucup-producao-2013a2016-2017-11-01-tecnica-patente.csv
7. br-capes-colsucup-producao-2013a2016-2017-11-01-artistica-outra.csv
8. br-capes-colsucup-producao-2013a2016-2017-11-01-bibliografica-livro.csv
9. br-capes-colsucup-producao-2013a2016-2017-11-01-tecnica-dprodu.csv
10. br-capes-colsucup-producao-2013a2016-2017-11-01-tecnica-deapli.csv
... mais 419 arquivos

Iniciando download de 429 arquivos...
[1/429] Já existe: producao/br-capes-colsucup-producao-2013a2016-2017-11-01-tecni

## Filtrando apenas registros da UFRJ dos dados abertos CAPES - 2013 em diante

In [838]:
#Função para selecionar documentos csv via expressões regulares e após um dado ano (e.g. 2013)
#A identificação do ano através do nome de arquivo desta função está adaptada à padronização de nomes dos arquivos da CAPES
#Valores de anos que iniciam quadriênios (2013, 2017, 2021) funcionarão normalmente. Outros anos poderão apresentar problemas
def filtrar_csvs_por_diretorio_regex_e_ano(
    caminho_do_diretorio,
    padroes_regex=None,
    ano_minimo=None, 
    busca_recursiva=False
):
    """
    Lista arquivos CSV em um diretório e os filtra com base em:
    1. Padrões de expressão regular (regex)
    2. Um ano mínimo encontrado no nome do arquivo.

    Args:
        caminho_do_diretorio (str): O caminho para o diretório base onde os CSVs estão.
        padroes_regex (list, optional): Uma lista de strings de expressões regulares.
                                        Arquivos devem corresponder a *qualquer um* desses padrões.
                                        Default é None (não aplica filtro regex).
        ano_minimo (int, optional): O ano mínimo (4 dígitos) para filtrar.
                                    Captura o primeiro grupo de 4 dígitos encontrado no nome do arquivo.
                                    Default é None (não aplica filtro de ano).
        busca_recursiva (bool, optional): Se True, a função buscará CSVs em subpastas também.
                                          Default é False.

    Returns:
        list: Uma lista de caminhos completos para os arquivos CSV que atendem a todos os critérios.

    Raises:
        ValueError: Se 'caminho_do_diretorio' não for encontrado ou se nenhum filtro for especificado.
    """
    if not os.path.exists(caminho_do_diretorio):
        print(f"Erro: O diretório '{caminho_do_diretorio}' não foi encontrado.")
        return []

    if not padroes_regex and ano_minimo is None:
        print("Aviso: Nenhum filtro (regex ou ano mínimo) foi especificado. Retornando todos os CSVs.")

    todos_csvs_encontrados = []

    # 1. Obter todos os arquivos CSV do diretório (e subpastas, se recursivo)
    if busca_recursiva:
        for root, _, files in os.walk(caminho_do_diretorio):
            for nome_arquivo in files:
                if nome_arquivo.lower().endswith(".csv"):
                    todos_csvs_encontrados.append(os.path.join(root, nome_arquivo))
    else:
        for nome_arquivo in os.listdir(caminho_do_diretorio):
            caminho_completo = os.path.join(caminho_do_diretorio, nome_arquivo)
            if os.path.isfile(caminho_completo) and nome_arquivo.lower().endswith(".csv"):
                todos_csvs_encontrados.append(caminho_completo)

    arquivos_filtrados_parcial = todos_csvs_encontrados

    # 2. Aplicar filtro por padrões Regex (se fornecidos)
    if padroes_regex:
        final_regex_filter = []
        regexes_compilados = [re.compile(p, re.IGNORECASE) for p in padroes_regex]

        for caminho_completo in arquivos_filtrados_parcial:
            nome_base = os.path.basename(caminho_completo)
            for regex in regexes_compilados:
                if regex.search(nome_base):
                    final_regex_filter.append(caminho_completo)
                    break
        arquivos_filtrados_parcial = final_regex_filter

    # 3. Aplicar filtro por ano mínimo (se fornecido)
    if ano_minimo is not None:
        final_year_filter = []
        # Regex para qualquer sequência de 4 dígitos - adaptado à padronização CAPES
        # O padrão (\d{4}) captura o ano. Valores de anos que iniciam quadriênios (2013, 2017, 2021)
        # funcionarão normalmente. Outros anos poderão apresentar problemas se o formato 'discentes-AAAA'
        # não for o primeiro e único lugar onde o ano pode estar.
        padrao_ano_capes = re.compile(r'(\d{4})') 

        for caminho_completo in arquivos_filtrados_parcial:
            nome_base = os.path.basename(caminho_completo)
            match = padrao_ano_capes.search(nome_base)
            
            if match:
                ano_str = match.group(1)
                try:
                    ano_int = int(ano_str)
                    if ano_int >= ano_minimo:
                        final_year_filter.append(caminho_completo)
                except ValueError:
                    print(f"Aviso: Não foi possível converter '{ano_str}' em ano inteiro para '{nome_base}'. Ignorando.")
        arquivos_filtrados_parcial = final_year_filter

    return arquivos_filtrados_parcial

In [839]:
#Função para obter caminhos utilizando o dicionário de diretórios definido no começo deste documento
def obter_caminho_completo(basedir_key, subdir_key, dirs=dirs):
    """
    Concatena os caminhos correspondentes a 'basedir_key' e 'subdir_key'
    do dicionário 'dirs' para formar um caminho completo.

    Args:
        basedir_key (str): A chave do diretório base em 'dirs'.
        subdir_key (str): A chave do subdiretório em 'dirs'.
        dirs (dict): Dicionário com o nome dos diretórios. Default: dirs.

    Returns:
        str: O caminho completo concatenado.

    Raises:
        ValueError: Se 'basedir_key' ou 'subdir_key' não forem chaves válidas em 'dirs'.
    """
    chaves_disponiveis = list(dirs.keys())

    if basedir_key not in dirs:
        raise ValueError(
            f"Erro: Chave '{basedir_key}' não encontrada em 'dirs' para basedir. "
            f"Chaves disponíveis: {chaves_disponiveis}"
        )

    if subdir_key not in dirs:
        raise ValueError(
            f"Erro: Chave '{subdir_key}' não encontrada em 'dirs' para subdir. "
            f"Chaves disponíveis: {chaves_disponiveis}"
        )
    
    # Obtém os valores de diretório do dicionário
    base_path = dirs[basedir_key]
    sub_path = dirs[subdir_key]

    # Concatena os caminhos usando os.path.join para compatibilidade entre sistemas
    caminho_final = os.path.join(base_path, sub_path)
    
    return caminho_final

In [840]:
#Junção das funções 'filtrar_csvs_por_diretorio_regex_e_ano()' e 'obter_caminho_completo()'
#Usada para obter uma lista com os arquivos csv de interesse a serem unificados em uma tabela única
def selecionar_csvs_capes(
    basedir_key,
    subdir_key,
    dirs=dirs,
    padroes_regex=None,
    ano_minimo=2013,
    busca_recursiva=False
):
    """
    Integra as funções para obter o caminho completo do diretório e filtrar arquivos CSV.

    Args:
        dirs (dict): Dicionário com o nome dos diretórios. Default: dirs.
        basedir_key (str): A chave do diretório base em 'dirs' (e.g., 'download_dir', 'ufrj_dir').
        subdir_key (str): A chave do subdiretório em 'dirs' (e.g., 'discentes', 'producao').
        padroes_regex (list, optional): Uma lista de strings de expressões regulares para filtrar nomes de arquivo.
                                        Arquivos devem corresponder a *qualquer um* desses padrões.
                                        Default é None (não aplica filtro regex).
        ano_minimo (int, optional): O ano mínimo (4 dígitos) para filtrar.
                                    Adapta-se ao padrão "discentes-AAAA" dos arquivos da CAPES.
                                    Valores de anos que iniciam quadriênios (2013, 2017, 2021) funcionarão
                                    normalmente. Outros anos poderão apresentar problemas se o formato não se encaixar.
                                    Default é None (não aplica filtro de ano).
        busca_recursiva (bool, optional): Se True, a função buscará CSVs em subpastas do diretório gerado.
                                          Default é False.

    Returns:
        list: Uma lista de caminhos completos para os arquivos CSV que atendem a todos os critérios.

    Raises:
        ValueError: Se 'basedir_key' ou 'subdir_key' forem inválidas, ou se o diretório final não existir.
    """
    try:
        # 1. Obter o caminho completo do diretório usando as chaves
        caminho_do_diretorio_completo = obter_caminho_completo(basedir_key, subdir_key)
        print(f"Buscando arquivos no diretório: {caminho_do_diretorio_completo}")

    except ValueError as e:
        print(f"Erro ao obter caminho do diretório: {e}")
        return []

    # 2. Filtrar os arquivos CSV dentro do diretório gerado
    arquivos_selecionados = filtrar_csvs_por_diretorio_regex_e_ano(
        caminho_do_diretorio=caminho_do_diretorio_completo,
        padroes_regex=padroes_regex,
        ano_minimo=ano_minimo,
        busca_recursiva=busca_recursiva
    )

    return arquivos_selecionados

In [841]:
#Função para processar e salvar csvs relacionados em um único arquivo
def fundir_lista_csvs(
        lista_caminhos_csv,
        colunas_desejadas,
        diretorio_saida,
        condicao_filtro_funcao = None,
        nome_arquivo_saida="saida_otimizada_lista.csv",
        chunk_size=10000,
        colunas_para_int64=None
        ):
    """
    Processa uma lista de arquivos CSV, extraindo colunas e linhas específicas de forma otimizada para RAM
    usando leitura em chunks. Inclui opção para converter colunas para tipo Int64 (inteiro com nulos)
    e lida com colunas ausentes em arquivos CSV.

    Args:
        lista_caminhos_csv (list): Uma lista de strings, onde cada string é o caminho completo para um arquivo CSV.
        colunas_desejadas (list): Uma lista de nomes de colunas a serem extraídas.
                                   A coluna usada para o filtro (se houver) deve ser incluída explicitamente aqui
                                   se você quiser que ela apareça no resultado final.
        condicao_filtro_funcao (function): Uma função que recebe uma linha (como Series do pandas)
                                           e retorna True se a linha deve ser incluída, False caso contrário.
                                           Default: None (sem filtragem)
        diretorio_saida (str): O caminho para o diretório onde o arquivo de saída será salvo.
        nome_arquivo_saida (str): O nome do arquivo CSV de saída (ex: "meu_arquivo.csv").
        chunk_size (int): O número de linhas a serem lidas por vez de cada arquivo CSV.
        colunas_para_int64 (list, optional): Uma lista de nomes de colunas que devem ser convertidas
                                              para o tipo 'Int64' (inteiro com suporte a nulos) antes de salvar.
                                              Isso evita a adição de '.0' em IDs. Default é None.
    """
    primeiro_arquivo = True
    coluna_filtro = None # Manter para compatibilidade, mas não será mais usado para adicionar/remover colunas

    caminho_completo_saida = os.path.join(diretorio_saida, nome_arquivo_saida)

    if diretorio_saida and not os.path.exists(diretorio_saida):
        os.makedirs(diretorio_saida, exist_ok=True)
        print(f"Diretório de saída criado: {diretorio_saida}")

    # A lógica de modificação de colunas_desejadas e colunas_para_salvar foi removida
    colunas_para_ler = list(colunas_desejadas) # Agora 'colunas_para_ler' é simplesmente 'colunas_desejadas'
    colunas_para_salvar = list(colunas_desejadas) # E 'colunas_para_salvar' também

    # A lógica de remoção da coluna de filtro também foi removida
    # if remove_filter_col e a lógica associada não estão mais presentes

    for caminho_completo_arquivo_entrada in lista_caminhos_csv:
        if not os.path.exists(caminho_completo_arquivo_entrada):
            print(f"Aviso: Arquivo não encontrado - {caminho_completo_arquivo_entrada}. Pulando...")
            continue
        if not caminho_completo_arquivo_entrada.lower().endswith(".csv"):
            print(f"Aviso: Ignorando arquivo não CSV - {caminho_completo_arquivo_entrada}.")
            continue

        print(f"Processando {caminho_completo_arquivo_entrada}...")

        # Ler o cabeçalho para verificar as colunas presentes
        try:
            df_header = pd.read_csv(caminho_completo_arquivo_entrada, sep=';', encoding='latin1', nrows=0)
            colunas_presentes_no_arquivo = df_header.columns.tolist()
        except Exception as e:
            print(f"Erro ao ler o cabeçalho do arquivo {caminho_completo_arquivo_entrada}: {e}. Pulando...")
            continue

        # Identificar colunas a serem lidas que realmente existem no arquivo
        colunas_a_realmente_ler = [col for col in colunas_para_ler if col in colunas_presentes_no_arquivo]
        colunas_ausentes_neste_arquivo = [col for col in colunas_para_ler if col not in colunas_presentes_no_arquivo]

        if colunas_ausentes_neste_arquivo:
            print(f"Aviso: As seguintes colunas desejadas não foram encontradas em '{caminho_completo_arquivo_entrada}': {', '.join(colunas_ausentes_neste_arquivo)}. Elas serão adicionadas como vazias.")

        read_csv_args = {
            'sep': ';',
            'encoding': 'latin1',
            'usecols': colunas_a_realmente_ler,
            'chunksize': chunk_size
        }

        try:
            for chunk in pd.read_csv(caminho_completo_arquivo_entrada, **read_csv_args):
                # Adicionar colunas ausentes no chunk com valores NaN (vazios)
                for col in colunas_ausentes_neste_arquivo:
                    chunk[col] = pd.NA

                if condicao_filtro_funcao:
                    df_filtrado = chunk[chunk.apply(condicao_filtro_funcao, axis=1)]
                else:
                    df_filtrado = chunk

                # Certificar-se de que todas as colunas desejadas estão presentes antes de selecionar
                # e manter a ordem das colunas definidas em colunas_para_salvar
                df_final = pd.DataFrame(columns=colunas_para_salvar)
                for col in colunas_para_salvar:
                    if col in df_filtrado.columns:
                        df_final[col] = df_filtrado[col]
                    else:
                        df_final[col] = pd.NA

                # NOVO PASSO: Converter as colunas especificadas para Int64Dtype
                if colunas_para_int64:
                    for col in colunas_para_int64:
                        if col in df_final.columns:
                            try:
                                df_final[col] = pd.to_numeric(df_final[col], errors='coerce')
                                df_final[col] = df_final[col].astype('Int64')
                            except Exception as e:
                                print(f"Aviso: Não foi possível converter a coluna '{col}' para Int64 no chunk. Erro: {e}")
                        else:
                            print(f"Aviso: Coluna '{col}' não encontrada no DataFrame para conversão para Int64 neste chunk.")

                if primeiro_arquivo:
                    df_final.to_csv(caminho_completo_saida, mode='w', index=False)
                    primeiro_arquivo = False
                else:
                    df_final.to_csv(caminho_completo_saida, mode='a', header=False, index=False)
        except Exception as e:
            print(f"Erro ao ler ou processar o arquivo {caminho_completo_arquivo_entrada} em chunks: {e}")
            continue

    print(f"Processamento concluído. Saída salva em {caminho_completo_saida}")

### Discentes 

In [842]:
#Definição de filtro 
def filtro_ufrj_sigla(linha):
    """
    Verifica se a linha atende aos critérios de filtro:
    - 'SG_ENTIDADE_ENSINO' é 'UFRJ'.

    Args:
        linha (pd.Series): Uma linha do DataFrame.

    Returns:
        bool: True se a linha atende aos critérios, False caso contrário.
    """
    condicao_entidade = linha['SG_ENTIDADE_ENSINO'].strip().upper() == 'UFRJ'

    return condicao_entidade 

In [843]:
#Obtendo lista de arquivos csv com ano de referencia = 2013 ou maior (sem filtragem por regex)
discentes_csvs = selecionar_csvs_capes('download_dir', 'discentes')

Buscando arquivos no diretório: capes_csv_files/discentes


In [844]:
discentes_csvs

['capes_csv_files/discentes/br-capes-colsucup-discentes-2021-2025-03-31.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2013-2021-03-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2022-2025-03-31.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2014-2021-03-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2020-2023-12-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2016-2021-03-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2023-2025-03-31.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2018-2023-12-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2015-2021-03-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2019-2023-12-01.csv',
 'capes_csv_files/discentes/br-capes-colsucup-discentes-2017-2023-12-01.csv']

In [845]:
fundir_lista_csvs(discentes_csvs, 
                colunas_desejadas=['AN_BASE', 'ID_PESSOA', 'CD_PROGRAMA_IES', 'NM_DISCENTE', 'DS_TIPO_NACIONALIDADE_DISCENTE', 
                                    'NM_PAIS_NACIONALIDADE_DISCENTE', 'AN_NASCIMENTO_DISCENTE',
                                    'DS_GRAU_ACADEMICO_DISCENTE', 'ST_INGRESSANTE', 'NM_SITUACAO_DISCENTE',
                                    'QT_MES_TITULACAO', 'SG_ENTIDADE_ENSINO'], 
                                    condicao_filtro_funcao=filtro_ufrj_sigla,
                                    diretorio_saida=filtered_dir,
                                    nome_arquivo_saida='discentes.csv'
                )

Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2021-2025-03-31.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2013-2021-03-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2022-2025-03-31.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2014-2021-03-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2020-2023-12-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2016-2021-03-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2023-2025-03-31.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2018-2023-12-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2015-2021-03-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2019-2023-12-01.csv...
Processando capes_csv_files/discentes/br-capes-colsucup-discentes-2017-2023-12-01.csv...
Processamento concluí

### Docentes

In [846]:
docentes_csvs = selecionar_csvs_capes('download_dir', 'docentes')

Buscando arquivos no diretório: capes_csv_files/docentes


In [847]:
docentes_csvs

['capes_csv_files/docentes/br-capes-colsucup-docente-2023-2025-03-31.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2013-2023-08-01.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2019-2021-11-10.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2022-2025-03-31.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2015-2023-08-01.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2020-2021-11-10.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2017-2021-11-10.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2021-2025-03-31.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2016-2023-08-01.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2014-2023-08-01.csv',
 'capes_csv_files/docentes/br-capes-colsucup-docente-2018-2021-11-10.csv']

In [848]:
fundir_lista_csvs(docentes_csvs, 
                  colunas_desejadas=['AN_BASE', 'ID_PESSOA', 'CD_PROGRAMA_IES',
                  'NM_DOCENTE', 'AN_NASCIMENTO_DOCENTE', 'DS_TIPO_NACIONALIDADE_DOCENTE',
                  'NM_PAIS_NACIONALIDADE_DOCENTE', 'DS_CATEGORIA_DOCENTE', 
                  'DS_TIPO_VINCULO_DOCENTE_IES', 'DS_REGIME_TRABALHO',
                  'CD_CAT_BOLSA_PRODUTIVIDADE', 'NM_GRAU_TITULACAO', 'SG_ENTIDADE_ENSINO'], 
                  condicao_filtro_funcao=filtro_ufrj_sigla,
                  diretorio_saida=filtered_dir,
                  nome_arquivo_saida='docentes.csv')

Processando capes_csv_files/docentes/br-capes-colsucup-docente-2023-2025-03-31.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2013-2023-08-01.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2019-2021-11-10.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2022-2025-03-31.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2015-2023-08-01.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2020-2021-11-10.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2017-2021-11-10.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2021-2025-03-31.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2016-2023-08-01.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2014-2023-08-01.csv...
Processando capes_csv_files/docentes/br-capes-colsucup-docente-2018-2021-11-10.csv...
Processamento concluído. Saída salva em ufrj_data/doce

### Programas

In [849]:
programas_csvs = selecionar_csvs_capes('download_dir', 'programas', padroes_regex=[r"br-capes-colsucup-prog"])

Buscando arquivos no diretório: capes_csv_files/programas


In [850]:
programas_csvs

['capes_csv_files/programas/br-capes-colsucup-prog-2017-2021-11-10.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2013a2016-2020-06-12_2013.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2019-2021-11-10.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2022-2025-03-31.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2021-2025-03-31.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2023-2025-03-31.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2020-2021-11-10.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2018-2021-11-10.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2013a2016-2020-06-12_2015.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2013a2016-2020-06-12_2016.csv',
 'capes_csv_files/programas/br-capes-colsucup-prog-2013a2016-2020-06-12_2014.csv']

In [851]:
fundir_lista_csvs(programas_csvs, ['AN_BASE', 'CD_PROGRAMA_IES', 'NM_PROGRAMA_IES', 'NM_GRANDE_AREA_CONHECIMENTO', 
                                    'NM_GRAU_PROGRAMA', 'CD_CONCEITO_PROGRAMA', 'ANO_INICIO_PROGRAMA', 'AN_INICIO_PROGRAMA',
                                    'AN_INICIO_CURSO', 'IN_REDE', 'DS_SITUACAO_PROGRAMA',
                                    'CD_AREA_AVALIACAO', 'NM_AREA_AVALIACAO', 'NM_MODALIDADE_PROGRAMA', 'SG_ENTIDADE_ENSINO',
                                   ], 
                                   condicao_filtro_funcao=filtro_ufrj_sigla,
                                   diretorio_saida=filtered_dir,
                                   nome_arquivo_saida='programas.csv'
                                   )

Processando capes_csv_files/programas/br-capes-colsucup-prog-2017-2021-11-10.csv...
Aviso: As seguintes colunas desejadas não foram encontradas em 'capes_csv_files/programas/br-capes-colsucup-prog-2017-2021-11-10.csv': ANO_INICIO_PROGRAMA. Elas serão adicionadas como vazias.
Processando capes_csv_files/programas/br-capes-colsucup-prog-2013a2016-2020-06-12_2013.csv...
Aviso: As seguintes colunas desejadas não foram encontradas em 'capes_csv_files/programas/br-capes-colsucup-prog-2013a2016-2020-06-12_2013.csv': AN_INICIO_PROGRAMA. Elas serão adicionadas como vazias.
Processando capes_csv_files/programas/br-capes-colsucup-prog-2019-2021-11-10.csv...
Aviso: As seguintes colunas desejadas não foram encontradas em 'capes_csv_files/programas/br-capes-colsucup-prog-2019-2021-11-10.csv': ANO_INICIO_PROGRAMA. Elas serão adicionadas como vazias.
Processando capes_csv_files/programas/br-capes-colsucup-prog-2022-2025-03-31.csv...
Aviso: As seguintes colunas desejadas não foram encontradas em 'capes

### Produção (artigos de periódicos por autor)

In [852]:
producao_csvs = selecionar_csvs_capes('download_dir', 'producao', padroes_regex=[r"br-capes-colsucup-prod-autor-.*bibliografica-artpe"])

Buscando arquivos no diretório: capes_csv_files/producao


In [853]:
producao_csvs

['capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-03-31-bibliografica-artpe-2022.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2013a2016-2017-03-01-bibliografica-artpe.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2019.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-03-31-bibliografica-artpe-2023.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-03-31-bibliografica-artpe-2021.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2018.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2020.csv',
 'capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2017.csv']

In [854]:
def filtro_producao(linha):
    """
    Verifica se a linha atende aos critérios de filtro:
    - 'SG_ENTIDADE_ENSINO' é 'UFRJ'.
    - 'ID_PESSOA_DOCENTE' não é nulo.
    - 'ID_PESSOA_DISCENTE' não é nulo.
    - 'NM_NIVEL_DISCENTE' está em algum nível que corresponda a alunos de pós.

    Args:
        linha (pd.Series): Uma linha do DataFrame.

    Returns:
        bool: True se a linha atende aos critérios, False caso contrário.
    """
    condicao_entidade = linha['SG_ENTIDADE_ENSINO'].strip().upper() == 'UFRJ'
    condicao_discente_nao_nulo = pd.notna(linha['ID_PESSOA_DISCENTE'])
    condicao_docente_nao_nulo = pd.notna(linha['ID_PESSOA_DOCENTE'])

    condicao_nivel_discente = True #Se a linha não estiver se referindo a discente, isso deve ser verdadeiro para não excluir o registro

    if condicao_discente_nao_nulo:
        nm_nivel_discente = str(linha.get('NM_NIVEL_DISCENTE', '')).strip().upper()
        condicao_nivel_discente = (nm_nivel_discente in ['MESTRADO', 'DOUTORADO', 'MESTRADO PROFISSIONAL', 'DOUTORADO PROFISSIONAL']) #Só inclui alunos de pós 
    
    return condicao_entidade and (condicao_discente_nao_nulo or condicao_docente_nao_nulo) and condicao_nivel_discente

In [855]:
fundir_lista_csvs(producao_csvs, 
                colunas_desejadas=['AN_BASE', 'CD_PROGRAMA_IES', 'NM_PROGRAMA_IES', 'ID_ADD_PRODUCAO_INTELECTUAL', 
                                   'ID_PESSOA_DOCENTE', 'ID_PESSOA_DISCENTE', 'TP_AUTOR', 
                                   'NM_TP_CATEGORIA_DOCENTE', 'NM_NIVEL_DISCENTE', 'SG_ENTIDADE_ENSINO'], 
                condicao_filtro_funcao=filtro_producao, 
                diretorio_saida=filtered_dir,
                nome_arquivo_saida='producao.csv',
                colunas_para_int64= ['ID_PESSOA_DOCENTE', 'ID_PESSOA_DISCENTE'] ,
               )

Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-03-31-bibliografica-artpe-2022.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2013a2016-2017-03-01-bibliografica-artpe.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2019.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-03-31-bibliografica-artpe-2023.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2021a2024-2025-03-31-bibliografica-artpe-2021.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2018.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2020.csv...
Processando capes_csv_files/producao/br-capes-colsucup-prod-autor-2017a2020-2022-05-31-bibliografica-artpe-2017.csv...
Processamento concluído. Saída salva em ufrj_data/pro

## Processamento/limpeza dos dados da ufrj

- Discentes:
    - Geração da coluna de gênero/remoção da coluna de nome
    - Converter ST_INGRESSANTE para booleano

- Docentes: 
    - Geração da coluna de gênero/remoção da coluna de nome
    - Limpeza campo CD_CAT_BOLSA_PRODUTIVIDADE (remover os NA e SR)

- Pessoas:
    - Pegar os campos de docentes e discentes que não mudam ao longo dos anos

- Programas:
    - Campo IN_REDE convertido para booleano
    - Substituir conceito 'A' ('Ausente') por 0
    - Fundir colunas ANO_INICIO_PROGRAMA (2013-2016) e AN_INICIO_PROGRAMA (2017 em diante)
    - Gerar a tabela 'programa' (informações que não mudam ao longo dos anos)
    - Gerar a tabela 'ano_programa', contendo apenas as informações sobre os programas que podem mudar ao longo dos anos
    - Gerar a tabela 'cursos', com a relação entre CD_PROGRAMA_IES, NM_GRAU_PROGRAMA e ANO_INICIO_CURSO
        - Separar os valores separados por barra em NM_GRAU_PROGRAMA (e.g. MESTRADO/DOUTORADO) e ANO_INICIO_CURSO (e.g. 1981/2001)
        - Adicionar outras linhas normalmente

- Produção:
    - Unir as colunas ID_PESSOA_DOCENTE e ID_PESSOA_DISCENTE em uma só

- Todas:
    - Remover as colunas usadas para a filtragem dos dados
    - Quando estiver presente, substituir o ID_PESSOA por um hash (ID_PESSOA + salt): Salvar a relação ID_PESSOA e salt em uma db sqlite separada

In [856]:
#Importando dicionário de gêneros
with open('aux/dicionario_generos.json', 'r') as f: 
    dicionario_generos = json.load(f)

dicionario_generos

{'AALINE': 'F',
 'AILINE': 'F',
 'ALEINE': 'F',
 'ALIINE': 'F',
 'ALINE': 'F',
 'ALINER': 'F',
 'ALINHE': 'F',
 'ALINNE': 'F',
 'ALYNE': 'F',
 'ALYNNE': 'F',
 'AYLINE': 'F',
 'EALINE': 'F',
 'ELEINE': 'F',
 'ELINE': 'F',
 'ELINER': 'F',
 'ELINNE': 'F',
 'ELYNE': 'F',
 'EULINE': 'F',
 'HALINE': 'F',
 'HALYNE': 'F',
 'HELEINE': 'F',
 'HELINE': 'F',
 'HELYNE': 'F',
 'IALINE': 'F',
 'ILEINE': 'F',
 'ILINE': 'F',
 'LEINE': 'F',
 'LEINER': 'F',
 'LEYNE': 'F',
 'LINE': 'F',
 'LINER': 'F',
 'LUEINE': 'F',
 'LUINE': 'F',
 'LUYNE': 'F',
 'LYNE': 'F',
 'LYNNE': 'F',
 'OLINE': 'F',
 'UELINE': 'F',
 'AARAO': 'M',
 'ARAAO': 'M',
 'ARAO': 'M',
 'AARON': 'M',
 'AHARON': 'M',
 'AROM': 'M',
 'ARON': 'M',
 'ARYON': 'M',
 'HARON': 'M',
 'ABA': 'F',
 'ADA': 'F',
 'ADAH': 'F',
 'ADAR': 'F',
 'ADHA': 'F',
 'HADA': 'F',
 'ABADE': 'M',
 'ABADI': 'M',
 'ABADIR': 'M',
 'ABADIA': 'F',
 'ABADIAS': 'M',
 'ABADIO': 'M',
 'ABAETE': 'F',
 'ABETE': 'F',
 'ADETE': 'F',
 'ABD': 'M',
 'ABDA': 'F',
 'ADDA': 'F',
 'ABDAEL':

In [857]:
# Define a função auxiliar que será aplicada a cada nome completo
def genero_baseado_no_primeiro_nome(nome_completo: str,
                    dicionario_generos: dict) -> str:
    
    # Garante que é uma string, útil para lidar com NaNs ou outros tipos
    nome_completo_str = str(nome_completo) 
    
    primeiro_nome = nome_completo_str.split(' ')[0].strip().upper()
    
    # Usa .get() para retornar 'D' (Desconhecido) se o nome não for encontrado
    return dicionario_generos.get(primeiro_nome, 'D')


def adicionar_coluna_genero(df,
                                 coluna_nome_completo: str, 
                                 dicionario_generos: dict,
                                 coluna_genero: str = 'TP_SEXO',
                       ):
    """
    Extrai o primeiro nome de uma coluna, consulta um dicionário de gêneros baseados em primeiros nomes
    e retorna os gêneros correspondentes em uma nova coluna, usando df.apply().

    Args:
        df (pd.DataFrame): O DataFrame de entrada.
        coluna_nome_completo (str): O nome da coluna no DataFrame que contém os nomes completos.
        dicionario_generos (dict): Um dicionário onde as chaves são os primeiros nomes (em maiúsculas)
                                   e os valores são os gêneros ('F', 'M', 'Desconhecido', etc.).
        coluna_genero (str): Nome da nova coluna com o genero inferido com base no primeiro nome. O padrão é 'GN_PESSOA'.

    Returns:
        pd.DataFrame: O DataFrame original com uma nova coluna .
    """
    df[coluna_genero] = df[coluna_nome_completo].apply(genero_baseado_no_primeiro_nome, 
                                                       dicionario_generos=dicionario_generos)
    
    return df

In [858]:
#Criando diretório de saída dos arquivos processados
try:
    # Cria o diretório
    # 'exist_ok=True' é crucial: se o diretório já existir, ele não levantará um erro (FileExistsError)
    os.makedirs(processed_dir, exist_ok=True)
    print(f"Diretório '{processed_dir}' criado com sucesso ou já existente.")
except OSError as e:
    # Trata outros possíveis erros do sistema operacional (permissões, nomes inválidos, etc.)
    print(f"Erro ao criar o diretório '{processed_dir}': {e}")

Diretório 'sucupira_painel' criado com sucesso ou já existente.


### Substituição de IDs por hashes (anonimização dos dados)

A idéia aqui é:

- Pegar todos os ids únicos dos dados da CAPES
- Gerar um salt aleatório para cada um
- Fazer o hashing do id_original + salt
- Salvar a correspondência entre id_original, salt e hash em uma db sqlite encriptada 
    - Essa db servirá como armazenamento persistente para cenários (improváveis) em que precisemos retornar a algum id_original a partir do hash
- Gerar um dicionário com a relação entre id_original e hash
    - Com base no dicionário, substituir o id_original pelo hash nas tabelas para anonimizar os dados

In [859]:
# --- Configuração ---
# Carrega as variáveis de ambiente do arquivo .env primeiro.
# Usar override=True garante que as alterações no .env sejam lidas sem reiniciar o kernel.
load_dotenv(override=True) 

#Salvando em variáveis as informações do dotenv sobre a database que irá guardar os id_pessoa e seus respectivos salts e hashes:
DATABASE_FILEPATH = os.getenv('DATABASE_FILEPATH') #Caminho para a database
DATABASE_PASSWORD = os.getenv('DATABASE_PASSWORD') #Senha da database


# --- Mensagens de Depuração da Configuração --
print(f"DEBUG: Caminho do arquivo do banco de dados: '{DATABASE_FILEPATH}'")
#print(f"DEBUG: DATABASE_PASSWORD carregado (primeiros caracteres): '{DATABASE_PASSWORD[:4]}...'") # Evita imprimir a senha completa
print(f"DEBUG: Número de núcleos da CPU detectados: {os.cpu_count()}")


# --- Função de Configuração do Banco de Dados ---
def setup_secure_database():
    """
    Configura e criptografa o banco de dados SQLite, se ele não existir.
    
    A tabela é genérica e pode armazenar hashes para qualquer tipo de ID.
    """
    print(f"\n--- Configurando Banco de Dados Seguro ---")
    print(f"DEBUG: Tentando conectar a: {DATABASE_FILEPATH}")
    try:
        # Usa a declaração 'with' para tratamento automático da conexão
        with sqlite.connect(DATABASE_FILEPATH) as conn: 
            conn.execute(f"PRAGMA key = '{DATABASE_PASSWORD}';")
            conn.execute("PRAGMA cipher_migrate;") # Garante que a criptografia mais recente seja aplicada ao banco sqlite
            
            cursor = conn.cursor()
            print("DEBUG: Tentando CRIAR TABELA SE NÃO EXISTIR 'identificadores_privados'.")
            cursor.execute('''
                CREATE TABLE IF NOT EXISTS identificadores_privados (
                    id_original TEXT PRIMARY KEY,
                    salt BLOB NOT NULL UNIQUE,
                    hash_public_fixo TEXT NOT NULL UNIQUE,
                    data_geracao TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    data_ultima_atualizacao TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                );
            ''')
            conn.commit() # Confirma explicitamente as alterações
            print(f"Banco de dados '{DATABASE_FILEPATH}' configurado e criptografado com sucesso.")
            
    except sqlite.Error as e:
        print(f"ERRO na configuração do banco de dados seguro: {e}")
        
# --- Funções Auxiliares para Hashing ---
def generate_salt(length=16):
    """Gera um salt aleatório."""
    return os.urandom(length)

def generate_hashed_id(id_and_salt_tuple):
    """
    Gera um hash SHA256 usando o ID original e um salt,
    recebendo-os como uma tupla para uso em paralelismo.
    """
    original_id, salt = id_and_salt_tuple # Desempacota a tupla
    
    id_bytes = str(original_id).encode('utf-8')
    combined = id_bytes + salt
    new_hash_public = hashlib.sha256(combined).hexdigest() #Faz o hash e o converte para string
    
    # Retorna a tupla completa (id_original, salt, hash)
    # que é o formato esperado para a inserção no DB.
    return (original_id, salt, new_hash_public)


# --- Lógica Principal (Paralelismo) ---
def get_or_create_hashed_ids_secure_parallel(original_ids_list, batch_size=5000, max_workers=os.cpu_count()): # batch_size é útil para inserção em lote
    results = {}
    ids_and_salts_for_parallel_hashing = [] 

    print(f"\n--- Iniciando Processamento Paralelo para {len(original_ids_list):,} IDs ---")

    # Passo 1: Pré-verificação sequencial de IDs existentes no banco de dados (fase de leitura)
    try:
        with sqlite.connect(DATABASE_FILEPATH) as conn:
            conn.execute(f"PRAGMA key = '{DATABASE_PASSWORD}';")
            cursor = conn.cursor()

            if original_ids_list:
                # Divide a lista para verificar IDs existentes para evitar cláusulas IN muito grandes
                chunk_size_check = 1000 # Ajuste conforme necessário
                for i in range(0, len(original_ids_list), chunk_size_check):
                    chunk = original_ids_list[i:i + chunk_size_check]
                    placeholders = ','.join('?' for _ in chunk)
                    cursor.execute(f'SELECT id_original, hash_public_fixo FROM identificadores_privados WHERE id_original IN ({placeholders})', chunk)
                    for row in cursor.fetchall():
                        original_id, hash_public_fixo = row
                        results[original_id] = hash_public_fixo
                
    except sqlite.Error as e:
        print(f"ERROR durante a leitura inicial do DB para IDs: {e}")
        
    for original_id in original_ids_list:
        if original_id not in results:
            ids_and_salts_for_parallel_hashing.append((original_id, generate_salt()))

    print(f"DEBUG: {len(results):,} IDs encontrados no DB. {len(ids_and_salts_for_parallel_hashing):,} IDs precisam de novos hashes (processamento paralelo).")

    # Passo 2: Paraleliza o hashing para novos IDs
    new_hashes_data = []
    if ids_and_salts_for_parallel_hashing:
        print(f"DEBUG: Iniciando hashing paralelo para {len(ids_and_salts_for_parallel_hashing):,} IDs...")
        parallel_start_time = time.time()

        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            # map submete tarefas e coleta resultados na ordem
            new_hashes_data = list(executor.map(generate_hashed_id, ids_and_salts_for_parallel_hashing))

        parallel_end_time = time.time()
        print(f"DEBUG: Hashing paralelo concluído! Levou {parallel_end_time - parallel_start_time:.2f} segundos.")
    else:
        print("DEBUG: Nenhum ID novo para fazer hashing em paralelo.")


    # Passo 3: Insere novos hashes no banco de dados sequencialmente (escrita em lote)
    if new_hashes_data:
        print(f"DEBUG: Iniciando inserção em lote sequencial para {len(new_hashes_data):,} novos hashes...")
        insert_start_time = time.time()

        try:
            with sqlite.connect(DATABASE_FILEPATH) as conn:
                conn.execute(f"PRAGMA key = '{DATABASE_PASSWORD}';")
                cursor = conn.cursor()

                for i in range(0, len(new_hashes_data), batch_size):
                    chunk_to_insert = new_hashes_data[i:i + batch_size]
                    cursor.executemany('''
                        INSERT OR IGNORE INTO identificadores_privados (id_original, salt, hash_public_fixo)
                        VALUES (?, ?, ?)
                    ''', chunk_to_insert)
                    conn.commit()

            print(f"DEBUG: Inserção em lote concluída! Levou {time.time() - insert_start_time:.2f} segundos.")

            # Re-leitura simples para garantir que todos os IDs estejam no dicionário `results`
            # Este passo é crucial para garantir que os hashes de IDs que já existiam sejam recuperados
            print("DEBUG: Fazendo uma leitura final para garantir que todos os hashes estejam no resultado.")

            # Obter a lista completa de IDs originais para a leitura final
            all_original_ids = [item[0] for item in new_hashes_data]

            if all_original_ids:
                chunk_size_check_final = 1000
                with sqlite.connect(DATABASE_FILEPATH) as conn:
                    conn.execute(f"PRAGMA key = '{DATABASE_PASSWORD}';")
                    cursor = conn.cursor()
                    for i in range(0, len(all_original_ids), chunk_size_check_final):
                        chunk = all_original_ids[i:i + chunk_size_check_final]
                        placeholders = ','.join('?' for _ in chunk)
                        cursor.execute(f'SELECT id_original, hash_public_fixo FROM identificadores_privados WHERE id_original IN ({placeholders})', chunk)
                        for row in cursor.fetchall():
                            original_id, hash_public_fixo = row
                            results[original_id] = hash_public_fixo

        except sqlite.Error as e:
            print(f"ERROR durante a inserção ou leitura final: {e}")
            
    print(f"--- Processamento de IDs Concluído ---")
    return results

DEBUG: Caminho do arquivo do banco de dados: 'id_hash_painel_gid.db'
DEBUG: Número de núcleos da CPU detectados: 12


In [860]:
#Configurando a db que irá receber os dados sobre o hashing
setup_secure_database()


--- Configurando Banco de Dados Seguro ---
DEBUG: Tentando conectar a: id_hash_painel_gid.db
DEBUG: Tentando CRIAR TABELA SE NÃO EXISTIR 'identificadores_privados'.
Banco de dados 'id_hash_painel_gid.db' configurado e criptografado com sucesso.


In [931]:
#Selecionando e renomeando as colunas de ids a serem hasheadas
id_pessoa_docente = pd.read_csv(f'{filtered_dir}/discentes.csv', usecols=['ID_PESSOA']).rename(columns = {'ID_PESSOA': 'ID'})
id_pessoa_discente = pd.read_csv(f'{filtered_dir}/docentes.csv', usecols=['ID_PESSOA']).rename(columns = {'ID_PESSOA': 'ID'})
id_producao = pd.read_csv(f'{filtered_dir}/producao.csv', usecols=['ID_ADD_PRODUCAO_INTELECTUAL']).rename(columns = {'ID_ADD_PRODUCAO_INTELECTUAL': 'ID'})
id_programa = pd.read_csv(f'{filtered_dir}/producao.csv', usecols=['CD_PROGRAMA_IES']).rename(columns = {'CD_PROGRAMA_IES': 'ID'})

In [932]:
#Concatenando e removendo duplicatas
ids_originais = pd.concat([id_pessoa_discente, id_pessoa_docente, id_producao, id_programa], ignore_index=True)['ID'].drop_duplicates()
ids_originais.is_unique #Checando se realmente só temos valores unicos

True

In [863]:
#Convertendo os dados em uma lista de strings (banco de dados espera que ID seja string)
lista_ids_originais_str = ids_originais.astype(str).tolist()

In [864]:
#Criando um dicionario que vai conter a relação entre os ids originais e seus hash correspondente
hash_lookup = get_or_create_hashed_ids_secure_parallel(lista_ids_originais_str)


--- Iniciando Processamento Paralelo para 142,214 IDs ---
DEBUG: 142,214 IDs encontrados no DB. 0 IDs precisam de novos hashes (processamento paralelo).
DEBUG: Nenhum ID novo para fazer hashing em paralelo.
--- Processamento de IDs Concluído ---


In [865]:
hash_lookup

{'100518': 'c99f2df8d15283f80f247e6489ee390a66dca0206b8db80bc7e09b7358e4eae9',
 '1009673': 'c325e5a041e42ddd0153f46010b01177ffcb08398aed29ef3edad1b64f487523',
 '1022949': 'eee64d0a3a4d15100e70e034672c41fb5c49ed7570abd49a8ce0fd727ebf8ac3',
 '10245': 'e72316c5b4bd3f216be4cf5428523a803ccc2d3bd9376906617bbadbbedb244c',
 '103106': '419362d216dc7aabd28f72697e170c99149f97197350a75f9863e97bd544a6bd',
 '103394': '007a956b7efc4797a3f3e6b01ad886b422a6dd564f61f76515839aa7a8efc149',
 '1041393': '976e1de9313a968c4600f58a261eb8ab4355e4f6a82c2ba86c6e0af7d5be3387',
 '104174': 'b0befb189ea498eca5e87ac82795d38c991be0b2a942a0f9eea2afb823035978',
 '105010': '5b31fb5ba13be363f9b0a200eb9a64533d9b8e4d7966de6facc98139cd45f58f',
 '105225': '442ea198da91d3f714d2cbd5686e2a2bf3170998e2f99fcc8cb0bfd51d4a99be',
 '106145': '1fdc85a94111db8923e53bdb0b2e4437af23736eccdaa41cb45fe82aae6d6a0a',
 '1061921': '3fe4934e6d814fc0660522bd197ecb5fb9dd7e988115091a3db426562686ac18',
 '10707': '7e7f95d472fd31bed0fa48d5aa453b74789edd

In [866]:
# Criando uma funcao que recebe uma série de ids devolve uma série modificada com os valores de hash
def converter_ids_para_hashes(serie_de_ids: pd.Series) -> pd.Series:
    """
    Converte uma Série de IDs para uma Série de hashes usando um dicionário de lookup.

    Args:
        serie_de_ids (pd.Series): Uma coluna (Série) do pandas contendo os IDs.

    Returns:
        pd.Series: Uma nova Série com os valores de hash correspondentes.
                   Retorna NaN para IDs não encontrados no dicionário.
    """
    # O método .astype(str) converte todos os ids para strings (que é como a db armazena os ids).
    # O método .map() faz todo o trabalho de procurar cada ID no dicionário.
    serie_de_hashes = serie_de_ids.astype(str).map(hash_lookup)
    return serie_de_hashes

### Discentes

In [867]:
#Importando dados dos discentes
discentes = pd.read_csv(f'{filtered_dir}/discentes.csv')
discentes.head()

,AN_BASE,ID_PESSOA,CD_PROGRAMA_IES,NM_DISCENTE,DS_TIPO_NACIONALIDADE_DISCENTE,NM_PAIS_NACIONALIDADE_DISCENTE,AN_NASCIMENTO_DISCENTE,DS_GRAU_ACADEMICO_DISCENTE,ST_INGRESSANTE,NM_SITUACAO_DISCENTE,QT_MES_TITULACAO,SG_ENTIDADE_ENSINO
0,2021,3353229,31001017100P2,BRENDO ARAUJO GOMES,BRASILEIRO,BRASIL,1994,DOUTORADO,SIM,MATRICULADO,0,UFRJ
1,2021,26505,31001017033P3,CLAUDIA BENITEZ LOGELO,BRASILEIRO,BRASIL,1973,DOUTORADO,NÃO,MATRICULADO,0,UFRJ
2,2021,1216819,31001017172P3,FRANCIANE PIMENTEL MELO,BRASILEIRO,BRASIL,1974,MESTRADO,NÃO,MATRICULADO,0,UFRJ
3,2021,794525,31001017020P9,GABRIELA MONTEZ HOLANDA DA SILVA,BRASILEIRO,BRASIL,1990,DOUTORADO,NÃO,MATRICULADO,0,UFRJ
4,2021,4485640,31001017134P4,DANIELLE BRODA DE VASCONCELLOS,BRASILEIRO,BRASIL,1991,MESTRADO PROFISSIONAL,SIM,MATRICULADO,0,UFRJ


In [868]:
#Gerando colunas com os hashes baseados nos ids originais
discentes['ID_PESSOA_HASH'] = converter_ids_para_hashes(discentes['ID_PESSOA'])
discentes['ID_PROGRAMA_HASH'] = converter_ids_para_hashes(discentes['CD_PROGRAMA_IES'])
discentes[['ID_PESSOA', 'ID_PESSOA_HASH', 'CD_PROGRAMA_IES', 'ID_PROGRAMA_HASH']]

,ID_PESSOA,ID_PESSOA_HASH,CD_PROGRAMA_IES,ID_PROGRAMA_HASH
0,3353229,7bb910b20b44c56ea76538f4f806b48d290427154bdeb5...,31001017100P2,f9526c24bcdd3d071a3e88e18ebe46befa704cc7569b67...
1,26505,f63dadcae091f1eb7b3dc5442bbba11356c860f3688908...,31001017033P3,57367b688df5a36cf5181f8f8fba4924cc7a019ad031ef...
2,1216819,bd74d0ed8dcbf0c20e5d9a212d041f52b4079484207bab...,31001017172P3,64763ebe8052066377a38b117092914552a2dfa24b1b77...
3,794525,53d10ed08e2895741e97ef0ac31fc085b989cdd6758aee...,31001017020P9,76a2aede8e4a14d924d311074569b9ebf8ce28ed6a28ac...
4,4485640,b1d3bad657449255e6051a55d3f31d20819104286f5c2f...,31001017134P4,5aa8044ac601aea3575fdadcaffddf2e8fb000f4db1c95...
...,...,...,...,...
158386,978622,299238338c1022aa003baeaf3de9e630f2c0b44fce1059...,31001017103P1,3f132af57a57b07c227c51c817b65c52aecaaf62cb51ed...
158387,999691,4d34260c6bee12525013c1d32e72cf8506320ba3527950...,31001017103P1,3f132af57a57b07c227c51c817b65c52aecaaf62cb51ed...
158388,1001943,fd61f2ef7d0ad33ebf43f250276a172c0123835fd3457a...,31001017103P1,3f132af57a57b07c227c51c817b65c52aecaaf62cb51ed...
158389,1030429,9cfc8e0b18892c405f95cd9c01f4bf43c2745bcb032a01...,31001017103P1,3f132af57a57b07c227c51c817b65c52aecaaf62cb51ed...


In [869]:
#Gerando a coluna de sexo com base no primeiro nome
discentes = adicionar_coluna_genero(discentes,
                        coluna_nome_completo='NM_DISCENTE',
                        dicionario_generos=dicionario_generos)
discentes[['NM_DISCENTE', 'TP_SEXO']].head()

,NM_DISCENTE,TP_SEXO
0,BRENDO ARAUJO GOMES,M
1,CLAUDIA BENITEZ LOGELO,F
2,FRANCIANE PIMENTEL MELO,F
3,GABRIELA MONTEZ HOLANDA DA SILVA,F
4,DANIELLE BRODA DE VASCONCELLOS,F


In [870]:
#Convertendo coluna ST_INGRESSANTE para booleano
mapeamento_booleano = {'SIM': True, 'NÃO': False}
discentes['ST_INGRESSANTE'] = discentes['ST_INGRESSANTE'].map(mapeamento_booleano)

In [871]:
#Mantendo apenas as colunas necessárias (que mudam ao longo do tempo)
discentes_final_cols = ['AN_BASE', 'ID_PESSOA_HASH', 'ID_PROGRAMA_HASH', 'DS_GRAU_ACADEMICO_DISCENTE', 'ST_INGRESSANTE', 'NM_SITUACAO_DISCENTE', 'QT_MES_TITULACAO']
discentes_final = discentes[discentes_final_cols]

In [872]:
#Visualizando dataframe final
discentes_final.head()

,AN_BASE,ID_PESSOA_HASH,ID_PROGRAMA_HASH,DS_GRAU_ACADEMICO_DISCENTE,ST_INGRESSANTE,NM_SITUACAO_DISCENTE,QT_MES_TITULACAO
0,2021,7bb910b20b44c56ea76538f4f806b48d290427154bdeb5...,f9526c24bcdd3d071a3e88e18ebe46befa704cc7569b67...,DOUTORADO,True,MATRICULADO,0
1,2021,f63dadcae091f1eb7b3dc5442bbba11356c860f3688908...,57367b688df5a36cf5181f8f8fba4924cc7a019ad031ef...,DOUTORADO,False,MATRICULADO,0
2,2021,bd74d0ed8dcbf0c20e5d9a212d041f52b4079484207bab...,64763ebe8052066377a38b117092914552a2dfa24b1b77...,MESTRADO,False,MATRICULADO,0
3,2021,53d10ed08e2895741e97ef0ac31fc085b989cdd6758aee...,76a2aede8e4a14d924d311074569b9ebf8ce28ed6a28ac...,DOUTORADO,False,MATRICULADO,0
4,2021,b1d3bad657449255e6051a55d3f31d20819104286f5c2f...,5aa8044ac601aea3575fdadcaffddf2e8fb000f4db1c95...,MESTRADO PROFISSIONAL,True,MATRICULADO,0


In [873]:
#Salvando dataframe final
discentes_final.to_csv(f'{processed_dir}/discentes.csv', index=False)

### Docentes

In [874]:
#Importando df filtrada
docentes = pd.read_csv(f'{filtered_dir}/docentes.csv')
docentes.head()

,AN_BASE,ID_PESSOA,CD_PROGRAMA_IES,NM_DOCENTE,AN_NASCIMENTO_DOCENTE,DS_TIPO_NACIONALIDADE_DOCENTE,NM_PAIS_NACIONALIDADE_DOCENTE,DS_CATEGORIA_DOCENTE,DS_TIPO_VINCULO_DOCENTE_IES,DS_REGIME_TRABALHO,CD_CAT_BOLSA_PRODUTIVIDADE,NM_GRAU_TITULACAO,SG_ENTIDADE_ENSINO
0,2023,91992,31001017003P7,MARAL MOSTAFAZADEHFARD,1983,ESTRANGEIRO,IRÃ,COLABORADOR,SERVIDOR PÚBLICO,INTEGRAL,NaN,DOUTORADO,UFRJ
1,2023,513065,31001017003P7,KATRIN GRIT GELFERT,1973,ESTRANGEIRO,ALEMANHA,PERMANENTE,SERVIDOR PÚBLICO,INTEGRAL,1C,DOUTORADO,UFRJ
2,2023,173737,31001017003P7,SERGIO AUGUSTO ROMANA IBARRA,1984,BRASILEIRO,BRASIL,PERMANENTE,SERVIDOR PÚBLICO,INTEGRAL,NaN,DOUTORADO,UFRJ
3,2023,1129065,31001017003P7,ISAIA NISOLI,1982,ESTRANGEIRO,ITÁLIA,PERMANENTE,SERVIDOR PÚBLICO,INTEGRAL,NaN,DOUTORADO,UFRJ
4,2023,939007,31001017003P7,SEYED HAMID HASSANZADEH HAFSHEJANI,1982,ESTRANGEIRO,IRÃ,PERMANENTE,SERVIDOR PÚBLICO,DEDICAÇÃO EXCLUSIVA,NaN,DOUTORADO,UFRJ


In [875]:
#Gerando colunas com os hashes baseados nos ids originais
docentes['ID_PESSOA_HASH'] = converter_ids_para_hashes(docentes['ID_PESSOA'])
docentes['ID_PROGRAMA_HASH'] = converter_ids_para_hashes(docentes['CD_PROGRAMA_IES'])
docentes[['ID_PESSOA', 'ID_PESSOA_HASH', 'CD_PROGRAMA_IES', 'ID_PROGRAMA_HASH']]

,ID_PESSOA,ID_PESSOA_HASH,CD_PROGRAMA_IES,ID_PROGRAMA_HASH
0,91992,fdc28fcda7a35ff6af814990b2cf3359d14b7f3e70c599...,31001017003P7,30185f75b10750b7cb0b5ce989024d18b968cafc54491f...
1,513065,c41c9a485703095e528d4b3a9c51acdedddcf0beb1eb7a...,31001017003P7,30185f75b10750b7cb0b5ce989024d18b968cafc54491f...
2,173737,9cbf98ec3b7778704c378cfdf6abb5479bd6b72aa3ea1f...,31001017003P7,30185f75b10750b7cb0b5ce989024d18b968cafc54491f...
3,1129065,d633605ecf1cebfd68e1fa6270cac4e773079b2f9a0a45...,31001017003P7,30185f75b10750b7cb0b5ce989024d18b968cafc54491f...
4,939007,ecbd9aa4455dc46b74da47b4475d98cac95e6be9060aee...,31001017003P7,30185f75b10750b7cb0b5ce989024d18b968cafc54491f...
...,...,...,...,...
38518,976903,1f75a058cbd1e782e17b8b6f86065884d1d822a845d1d5...,31001017066P9,6e7d80835f6a13c53f9d6df7fc4e9cfb1f89417b76a307...
38519,534954,7d3e4767c31aa1070ce22994c2c877aeb2ec8ef1ed2eb3...,31001017128P4,08305d430ccafdd9806767af3995119bd418453f8511c6...
38520,16509,59a8fdcc060e27a95cbf6fd70450eb39b565c485dff575...,31001017028P0,a6b84ac15b26c3b1424709e0f002f74370f4ea88471d07...
38521,709792,bd2af8d1cc35d5737d9534f4510df171bb6e6f7b241c0d...,31001017138P0,bc878a01c5092a9b05b43f5305454ac4fe7b7092b72e3b...


In [876]:
#Gerando a coluna de gênero com base no primeiro nome
docentes = adicionar_coluna_genero(docentes,
                        coluna_nome_completo='NM_DOCENTE',
                        dicionario_generos=dicionario_generos)
docentes[['NM_DOCENTE', 'TP_SEXO']].head()

,NM_DOCENTE,TP_SEXO
0,MARAL MOSTAFAZADEHFARD,D
1,KATRIN GRIT GELFERT,F
2,SERGIO AUGUSTO ROMANA IBARRA,M
3,ISAIA NISOLI,M
4,SEYED HAMID HASSANZADEH HAFSHEJANI,D


In [877]:
#limpeza do campo CD_CAT_PRODUTIVIDADE
docentes['CD_CAT_BOLSA_PRODUTIVIDADE'].unique()

array([nan, '1C', '1D', '1B', '1A', '2', 'SR'], dtype=object)

In [878]:
docentes['CD_CAT_BOLSA_PRODUTIVIDADE'] = docentes['CD_CAT_BOLSA_PRODUTIVIDADE'].replace([np.nan, 'SR'], pd.NA)
docentes['CD_CAT_BOLSA_PRODUTIVIDADE'].unique()

array([<NA>, '1C', '1D', '1B', '1A', '2'], dtype=object)

In [879]:
#Checando se só temos docentes permanentes
docentes['DS_CATEGORIA_DOCENTE'].unique()

array(['COLABORADOR', 'PERMANENTE', 'VISITANTE'], dtype=object)

In [880]:
#Mantendo apenas colunas necessárias (que mudam ao longo do tempo)
docentes_final_cols = ['AN_BASE', 'ID_PESSOA_HASH', 'CD_PROGRAMA_IES', 'DS_CATEGORIA_DOCENTE', 'DS_TIPO_VINCULO_DOCENTE_IES', 'DS_REGIME_TRABALHO', 'CD_CAT_BOLSA_PRODUTIVIDADE', 'NM_GRAU_TITULACAO']
docentes_final = docentes[docentes_final_cols]

In [881]:
#Visualizando resultado final
docentes_final.head()

,AN_BASE,ID_PESSOA_HASH,CD_PROGRAMA_IES,DS_CATEGORIA_DOCENTE,DS_TIPO_VINCULO_DOCENTE_IES,DS_REGIME_TRABALHO,CD_CAT_BOLSA_PRODUTIVIDADE,NM_GRAU_TITULACAO
0,2023,fdc28fcda7a35ff6af814990b2cf3359d14b7f3e70c599...,31001017003P7,COLABORADOR,SERVIDOR PÚBLICO,INTEGRAL,<NA>,DOUTORADO
1,2023,c41c9a485703095e528d4b3a9c51acdedddcf0beb1eb7a...,31001017003P7,PERMANENTE,SERVIDOR PÚBLICO,INTEGRAL,1C,DOUTORADO
2,2023,9cbf98ec3b7778704c378cfdf6abb5479bd6b72aa3ea1f...,31001017003P7,PERMANENTE,SERVIDOR PÚBLICO,INTEGRAL,<NA>,DOUTORADO
3,2023,d633605ecf1cebfd68e1fa6270cac4e773079b2f9a0a45...,31001017003P7,PERMANENTE,SERVIDOR PÚBLICO,INTEGRAL,<NA>,DOUTORADO
4,2023,ecbd9aa4455dc46b74da47b4475d98cac95e6be9060aee...,31001017003P7,PERMANENTE,SERVIDOR PÚBLICO,DEDICAÇÃO EXCLUSIVA,<NA>,DOUTORADO


In [882]:
#Salvando dataframe final
docentes_final.to_csv(f'{processed_dir}/docentes.csv', index=False)

### Pessoas

In [883]:
def padronizar_nomes_colunas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Renomeia colunas que terminam com '_DOCENTE' ou '_DISCENTE' para nomes padronizados.
    Ex: 'AN_NASCIMENTO_DOCENTE' -> 'AN_NASCIMENTO'
        'NM_PAIS_NACIONALIDADE_DISCENTE' -> 'NM_PAIS_NACIONALIDADE'

    Args:
        df (pd.DataFrame): O DataFrame a ser renomeado.

    Returns:
        pd.DataFrame: O DataFrame com as colunas renomeadas.
    """
    novo_mapeamento = {}
    for coluna in df.columns:
        if coluna.endswith('_DOCENTE'):
            novo_mapeamento[coluna] = coluna.replace('_DOCENTE', '')
        elif coluna.endswith('_DISCENTE'):
            novo_mapeamento[coluna] = coluna.replace('_DISCENTE', '')
        else:
            novo_mapeamento[coluna] = coluna # Mantém colunas sem sufixo inalteradas
    
    return df.rename(columns=novo_mapeamento)

In [884]:
#Padronizando nomes de colunas entre docentes e discentes e extraindo colunas da tabela 'pessoas'
pessoas_cols = ['AN_BASE', 'ID_PESSOA_HASH', 'NM', 'AN_NASCIMENTO', 'NM_PAIS_NACIONALIDADE', 'TP_SEXO']
pessoas_docentes = padronizar_nomes_colunas(docentes)[pessoas_cols]
pessoas_discentes = padronizar_nomes_colunas(discentes)[pessoas_cols]

In [885]:
pessoas = pd.concat([pessoas_discentes, pessoas_docentes])

In [886]:
pessoas

,AN_BASE,ID_PESSOA_HASH,NM,AN_NASCIMENTO,NM_PAIS_NACIONALIDADE,TP_SEXO
0,2021,7bb910b20b44c56ea76538f4f806b48d290427154bdeb5...,BRENDO ARAUJO GOMES,1994,BRASIL,M
1,2021,f63dadcae091f1eb7b3dc5442bbba11356c860f3688908...,CLAUDIA BENITEZ LOGELO,1973,BRASIL,F
2,2021,bd74d0ed8dcbf0c20e5d9a212d041f52b4079484207bab...,FRANCIANE PIMENTEL MELO,1974,BRASIL,F
3,2021,53d10ed08e2895741e97ef0ac31fc085b989cdd6758aee...,GABRIELA MONTEZ HOLANDA DA SILVA,1990,BRASIL,F
4,2021,b1d3bad657449255e6051a55d3f31d20819104286f5c2f...,DANIELLE BRODA DE VASCONCELLOS,1991,BRASIL,F
...,...,...,...,...,...,...
38518,2018,1f75a058cbd1e782e17b8b6f86065884d1d822a845d1d5...,MARILEIA FRANCO MARINHO INOUE,1960,BRASIL,F
38519,2018,7d3e4767c31aa1070ce22994c2c877aeb2ec8ef1ed2eb3...,JORGE PAES BARRETO MARCONDES DE SOUZA,1957,BRASIL,M
38520,2018,59a8fdcc060e27a95cbf6fd70450eb39b565c485dff575...,CARLOS MAGLUTA,1958,BRASIL,M
38521,2018,bd2af8d1cc35d5737d9534f4510df171bb6e6f7b241c0d...,MARCOS DANTAS LOUREIRO,1948,BRASIL,M


In [887]:
#Após inspeção visual do dataset, averiguamos que há inconsistências com relação a campos como gênero e nacionalidade ao longo dos anos
#Erros de preenchimento podem ter ocorrido, mas eventos como naturalização e mudança de nome social também parecem desempenhar um papel importante
print(f'Um único id_pessoa pode aparecer com mais de uma nacionalidade? {pessoas.drop_duplicates(['ID_PESSOA_HASH','NM_PAIS_NACIONALIDADE'])['ID_PESSOA_HASH'].duplicated().any()}')
print(f'Um único id_pessoa pode aparecer com mais de um gênero? {pessoas.drop_duplicates(['ID_PESSOA_HASH','TP_SEXO'])['ID_PESSOA_HASH'].duplicated().any()}')

Um único id_pessoa pode aparecer com mais de uma nacionalidade? True
Um único id_pessoa pode aparecer com mais de um gênero? True


In [888]:
#Independentemente das inconsistências terem sido derivadas de erros de preenchimento ou não, uma abordagem adequada para essa tabela seria
#manter apenas os registros do último ano amostrado por pessoa e permitir a atualização com base no ano mais recente com o django ORM depois (update_or_create()).
#Assim, teremos sempre as informações mais recentes daquela pessoa.

pessoas_ultimo_ano = pessoas.sort_values('AN_BASE', ascending=False).drop_duplicates('ID_PESSOA_HASH') #Organiza a tabela por ano (descendente) e mantém apenas a primeira ocorrência de cada ID_PESSOA

In [889]:
#Dando uma olhada nos dados
pessoas_ultimo_ano

,AN_BASE,ID_PESSOA_HASH,NM,AN_NASCIMENTO,NM_PAIS_NACIONALIDADE,TP_SEXO
85598,2023,a8a4678073a7e11b04ed8988234b08e71a790c8ce7a693...,BRENO GARCIA BRAZ DE SOUZA,1990,BRASIL,M
85599,2023,821abd3643a3f27e7cd68aeb27f61d4206a7c26b319040...,CRISTOVAO MODESTO DA SILVA,1960,BRASIL,M
85600,2023,b16699c08be4a80c34a403913be3285e109f900d7314ac...,DANIELLA SILVA OLIVEIRA,1989,BRASIL,F
85601,2023,a895944c008b88fec2488b0c126fb3b66153ce3ea7e70d...,ANDRESSA LIMA DE VASCONCELOS,1991,BRASIL,F
85602,2023,49caac9799559a43ab1a8da36be36ea9861ab067bf2c2a...,LUIZ FERNANDO VILLAR DE FIGUEIREDO,1997,BRASIL,M
...,...,...,...,...,...,...
6165,2013,03df16a1e547e210b185a0960cb85a5bf1ae4d3d7a108f...,MARCO ANTONIO CARON RUFFINO,1963,BRASIL,M
6226,2013,2309bcafadef754e6b2a9852adea516a3315a5ebb6cbf4...,ELI ROQUE DINIZ,1939,BRASIL,M
6395,2013,3b52801e3ec4c678052cb537aea1d157d5843c1a143647...,GERALDO LUIZ DOS REIS NUNES,1948,BRASIL,M
6662,2013,f7c71b5a35630db3f6e3136a3a9c1903ae65486a3cee2e...,GERLINDE AGATE PLATAIS BRASIL TEIXEIRA,1957,BRASIL,F


In [890]:
#Removendo colunas desnecessárias para a database
pessoas_final = pessoas_ultimo_ano.drop(columns=['AN_BASE','NM'])

#Limpando possíveis valores não informados do nome do país
pessoas_final['NM_PAIS_NACIONALIDADE'] = pessoas_final['NM_PAIS_NACIONALIDADE'].replace('NÃO INFORMADO', pd.NA)

In [891]:
#Salvando dataframe final
pessoas_final.to_csv(f'{processed_dir}/pessoas.csv', index=False)

### Programas

In [892]:
#Importando df filtrada
programas = pd.read_csv(f'{filtered_dir}/programas.csv')
programas

,AN_BASE,CD_PROGRAMA_IES,NM_PROGRAMA_IES,NM_GRANDE_AREA_CONHECIMENTO,NM_GRAU_PROGRAMA,CD_CONCEITO_PROGRAMA,ANO_INICIO_PROGRAMA,AN_INICIO_PROGRAMA,AN_INICIO_CURSO,IN_REDE,DS_SITUACAO_PROGRAMA,CD_AREA_AVALIACAO,NM_AREA_AVALIACAO,NM_MODALIDADE_PROGRAMA,SG_ENTIDADE_ENSINO
0,2017,31001017005P0,ESTATÍSTICA,CIÊNCIAS EXATAS E DA TERRA,MESTRADO/DOUTORADO,5,NaN,1981.0,1981/2001,NÃO,EM FUNCIONAMENTO,1,MATEMÁTICA / PROBABILIDADE E ESTATÍSTICA,ACADÊMICO,UFRJ
1,2017,31001017162P8,SAÚDE PERINATAL,CIÊNCIAS DA SAÚDE,MESTRADO PROFISSIONAL,3,NaN,2015.0,2015,NÃO,EM FUNCIONAMENTO,16,MEDICINA II,PROFISSIONAL,UFRJ
2,2017,31001017158P0,ENGENHARIA DA NANOTECNOLOGIA,ENGENHARIAS,MESTRADO/DOUTORADO,4,NaN,2014.0,2014/2014,NÃO,EM FUNCIONAMENTO,12,ENGENHARIAS II,ACADÊMICO,UFRJ
3,2017,31001017156P8,ENSINO DE QUÍMICA,MULTIDISCIPLINAR,MESTRADO PROFISSIONAL,3,NaN,2014.0,2014,NÃO,EM FUNCIONAMENTO,46,ENSINO,PROFISSIONAL,UFRJ
4,2017,31001017151P6,NUTRIÇÃO CLÍNICA,CIÊNCIAS DA SAÚDE,MESTRADO PROFISSIONAL,3,NaN,2013.0,2013,NÃO,EM FUNCIONAMENTO,50,NUTRIÇÃO,PROFISSIONAL,UFRJ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1294,2014,31001017154P5,CIÊNCIA E TECNOLOGIA FARMACÊUTICA,CIÊNCIAS DA SAÚDE,MESTRADO PROFISSIONAL,3,2013.0,NaN,2013,NÃO,EM FUNCIONAMENTO,19,FARMÁCIA,PROFISSIONAL,UFRJ
1295,2014,31001017155P1,ENSINO DE HISTÓRIA,CIÊNCIAS HUMANAS,MESTRADO PROFISSIONAL,4,2014.0,NaN,2014,SIM,EM FUNCIONAMENTO,40,HISTÓRIA,PROFISSIONAL,UFRJ
1296,2014,31001017156P8,ENSINO DE QUÍMICA,MULTIDISCIPLINAR,MESTRADO PROFISSIONAL,3,2014.0,NaN,2014,NÃO,EM FUNCIONAMENTO,46,ENSINO,PROFISSIONAL,UFRJ
1297,2014,31001017157P4,ARTES DA CENA,"LINGÜÍSTICA, LETRAS E ARTES",MESTRADO,4,2014.0,NaN,2014,NÃO,EM FUNCIONAMENTO,11,ARTES / MÚSICA,ACADÊMICO,UFRJ


In [893]:
#Gerando colunas com os hashes baseados nos ids originais
programas['ID_PROGRAMA_HASH'] = converter_ids_para_hashes(programas['CD_PROGRAMA_IES'])
programas[['CD_PROGRAMA_IES', 'ID_PROGRAMA_HASH']]

,CD_PROGRAMA_IES,ID_PROGRAMA_HASH
0,31001017005P0,24e4155389ae312fc18cc8f84e9470892dd97aacbf6b84...
1,31001017162P8,b1437042c91b7c1df69f90bce30db83a9a74038c187595...
2,31001017158P0,4c7293f22405bb76daab3eb47a05d40a02edb1b16b54a7...
3,31001017156P8,a5e8ff9afe24a995222b925db858d1a020925a15b36af1...
4,31001017151P6,57063ae0b5389c020479940f5f35ce792b0d9f7f464bb2...
...,...,...
1294,31001017154P5,8d1026cd82f31ff928a7b7c9051c8284b668423f959e04...
1295,31001017155P1,99927fd51fa27594138bcccbaa247b9f230e1cc0d92304...
1296,31001017156P8,a5e8ff9afe24a995222b925db858d1a020925a15b36af1...
1297,31001017157P4,ad401b13681ece1c71974b2dfc2708f4f649864fae9457...


In [894]:
#Removendo o SG_ENTIDADE_ENSINO (todos os programas são da UFRJ, afinal)
programas = programas.drop(columns=['SG_ENTIDADE_ENSINO'])

In [895]:
#Checando valores do campo 'IN_REDE'
programas['IN_REDE'].unique()

array(['NÃO', 'SIM'], dtype=object)

In [896]:
#Convertendo valores para booleans
mapeamento_booleano = {'SIM': True, 'NÃO': False}
programas['IN_REDE'] = programas['IN_REDE'].map(mapeamento_booleano)

In [897]:
programas['IN_REDE'].unique()

array([False,  True])

In [898]:
#Checando conceitos atribuídos às PPGs
programas['CD_CONCEITO_PROGRAMA'].unique()

array(['5', '3', '4', '6', '7', 'A', '1'], dtype=object)

In [899]:
#Convertendo conceito 'A' (provavelmente 'Ausente') para 0
programas['CD_CONCEITO_PROGRAMA'] = programas['CD_CONCEITO_PROGRAMA'].replace('A', 0).astype('int') 

In [900]:
programas['CD_CONCEITO_PROGRAMA'].unique()

array([5, 3, 4, 6, 7, 0, 1])

In [901]:
#Checando colunas ANO_INICIO_PROGRAMA e AN_INICIO_PROGRAMA 
print(programas[['ANO_INICIO_PROGRAMA', 'AN_INICIO_PROGRAMA']])

      ANO_INICIO_PROGRAMA  AN_INICIO_PROGRAMA
0                     NaN              1981.0
1                     NaN              2015.0
2                     NaN              2014.0
3                     NaN              2014.0
4                     NaN              2013.0
...                   ...                 ...
1294               2013.0                 NaN
1295               2014.0                 NaN
1296               2014.0                 NaN
1297               2014.0                 NaN
1298               2014.0                 NaN

[1299 rows x 2 columns]


In [902]:
#Fundindo colunas ANO_INICIO_PROGRAMA e AN_INICIO_PROGRAMA 
#'AN_INICIO_PROGRAMA' convertido para int para evitar a permanência de floats graças aos nan anteriores
programas['AN_INICIO_PROGRAMA'] = programas['AN_INICIO_PROGRAMA'].fillna(programas['ANO_INICIO_PROGRAMA']).astype('int') 

In [903]:
print(programas[['ANO_INICIO_PROGRAMA', 'AN_INICIO_PROGRAMA']])

      ANO_INICIO_PROGRAMA  AN_INICIO_PROGRAMA
0                     NaN                1981
1                     NaN                2015
2                     NaN                2014
3                     NaN                2014
4                     NaN                2013
...                   ...                 ...
1294               2013.0                2013
1295               2014.0                2014
1296               2014.0                2014
1297               2014.0                2014
1298               2014.0                2014

[1299 rows x 2 columns]


In [904]:
#Removendo a coluna 'ANO_INICIO_PROGRAMA'
programas = programas.drop(columns=['ANO_INICIO_PROGRAMA'])

In [905]:
#Visualizando a tabela inteira
programas

,AN_BASE,CD_PROGRAMA_IES,NM_PROGRAMA_IES,NM_GRANDE_AREA_CONHECIMENTO,NM_GRAU_PROGRAMA,CD_CONCEITO_PROGRAMA,AN_INICIO_PROGRAMA,AN_INICIO_CURSO,IN_REDE,DS_SITUACAO_PROGRAMA,CD_AREA_AVALIACAO,NM_AREA_AVALIACAO,NM_MODALIDADE_PROGRAMA,ID_PROGRAMA_HASH
0,2017,31001017005P0,ESTATÍSTICA,CIÊNCIAS EXATAS E DA TERRA,MESTRADO/DOUTORADO,5,1981,1981/2001,False,EM FUNCIONAMENTO,1,MATEMÁTICA / PROBABILIDADE E ESTATÍSTICA,ACADÊMICO,24e4155389ae312fc18cc8f84e9470892dd97aacbf6b84...
1,2017,31001017162P8,SAÚDE PERINATAL,CIÊNCIAS DA SAÚDE,MESTRADO PROFISSIONAL,3,2015,2015,False,EM FUNCIONAMENTO,16,MEDICINA II,PROFISSIONAL,b1437042c91b7c1df69f90bce30db83a9a74038c187595...
2,2017,31001017158P0,ENGENHARIA DA NANOTECNOLOGIA,ENGENHARIAS,MESTRADO/DOUTORADO,4,2014,2014/2014,False,EM FUNCIONAMENTO,12,ENGENHARIAS II,ACADÊMICO,4c7293f22405bb76daab3eb47a05d40a02edb1b16b54a7...
3,2017,31001017156P8,ENSINO DE QUÍMICA,MULTIDISCIPLINAR,MESTRADO PROFISSIONAL,3,2014,2014,False,EM FUNCIONAMENTO,46,ENSINO,PROFISSIONAL,a5e8ff9afe24a995222b925db858d1a020925a15b36af1...
4,2017,31001017151P6,NUTRIÇÃO CLÍNICA,CIÊNCIAS DA SAÚDE,MESTRADO PROFISSIONAL,3,2013,2013,False,EM FUNCIONAMENTO,50,NUTRIÇÃO,PROFISSIONAL,57063ae0b5389c020479940f5f35ce792b0d9f7f464bb2...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1294,2014,31001017154P5,CIÊNCIA E TECNOLOGIA FARMACÊUTICA,CIÊNCIAS DA SAÚDE,MESTRADO PROFISSIONAL,3,2013,2013,False,EM FUNCIONAMENTO,19,FARMÁCIA,PROFISSIONAL,8d1026cd82f31ff928a7b7c9051c8284b668423f959e04...
1295,2014,31001017155P1,ENSINO DE HISTÓRIA,CIÊNCIAS HUMANAS,MESTRADO PROFISSIONAL,4,2014,2014,True,EM FUNCIONAMENTO,40,HISTÓRIA,PROFISSIONAL,99927fd51fa27594138bcccbaa247b9f230e1cc0d92304...
1296,2014,31001017156P8,ENSINO DE QUÍMICA,MULTIDISCIPLINAR,MESTRADO PROFISSIONAL,3,2014,2014,False,EM FUNCIONAMENTO,46,ENSINO,PROFISSIONAL,a5e8ff9afe24a995222b925db858d1a020925a15b36af1...
1297,2014,31001017157P4,ARTES DA CENA,"LINGÜÍSTICA, LETRAS E ARTES",MESTRADO,4,2014,2014,False,EM FUNCIONAMENTO,11,ARTES / MÚSICA,ACADÊMICO,ad401b13681ece1c71974b2dfc2708f4f649864fae9457...


In [906]:
#Assim como para pessoas, nós iremos manter na tabela 'programas' apenas a informação do último ano amostrado (mais atualizado)
programas_final_cols = ['AN_BASE', 'ID_PROGRAMA_HASH', 'NM_PROGRAMA_IES','NM_GRANDE_AREA_CONHECIMENTO','AN_INICIO_PROGRAMA', 'CD_AREA_AVALIACAO', 'NM_AREA_AVALIACAO', 'NM_MODALIDADE_PROGRAMA']
programas_final = programas[programas_final_cols].sort_values( #Seleciona colunas de interesse e ordena a tabela por ano (descendente)
                                                    'AN_BASE', 
                                                    ascending=False
                                                    ).drop_duplicates( #Mantém apenas a primeira ocorrência de 'CD_PROGRAMA_IES'
                                                    'ID_PROGRAMA_HASH'
                                                    ).drop(columns=['AN_BASE']) #Remove a coluna AN_BASE, que só foi necessária aqui para ordenar a tabela

In [907]:
#Vendo a tabela final
programas_final

,ID_PROGRAMA_HASH,NM_PROGRAMA_IES,NM_GRANDE_AREA_CONHECIMENTO,AN_INICIO_PROGRAMA,CD_AREA_AVALIACAO,NM_AREA_AVALIACAO,NM_MODALIDADE_PROGRAMA
623,7858d8d2adcb93f7a1e161c0b9f6f689b4ba33e5c2bb7f...,ECONOMIA POLÍTICA INTERNACIONAL,CIÊNCIAS HUMANAS,2009,39,CIÊNCIA POLÍTICA E RELAÇÕES INTERNACIONAIS,ACADÊMICO
622,425a10f6b0387c117d48bc5982ef619eec0907a816baf5...,ENGENHARIA BIOMÉDICA,ENGENHARIAS,1971,14,ENGENHARIAS IV,ACADÊMICO
621,e7609a056b478ced4fe1f1c1824ff40bdf639dca15cce1...,EDUCAÇÃO,CIÊNCIAS HUMANAS,1972,38,EDUCAÇÃO,ACADÊMICO
620,d18b58c9a5dbd5a12afe58c59f62c63f87e7e544156b24...,ENGENHARIA METALÚRGICA E DE MATERIAIS,ENGENHARIAS,1967,12,ENGENHARIAS II,ACADÊMICO
719,76cb482e9ad97378c43defce7dbe706899ff4a7e395fc9...,FÍSICA,CIÊNCIAS EXATAS E DA TERRA,1972,3,ASTRONOMIA / FÍSICA,ACADÊMICO
...,...,...,...,...,...,...,...
699,d305e32a8bf5090fe91c2e37fbfc8a71789bb18a79e4c3...,HISTÓRIA DAS CIÊNCIAS E DAS TÉCNICAS E EPISTEM...,MULTIDISCIPLINAR,2005,45,INTERDISCIPLINAR,ACADÊMICO
700,8a35a5557bb9b17aee3c695f83229c77c0c07b7bd6f603...,GEOCIÊNCIAS: PATRIMÔNIO GEOPALEONTOLÓGICO,CIÊNCIAS EXATAS E DA TERRA,2015,5,GEOCIÊNCIAS,ACADÊMICO
602,e616a4a68438a5ed799191bc46c22e711cc052072e2b60...,ENSINO DE MATEMÁTICA,MULTIDISCIPLINAR,2006,46,ENSINO,ACADÊMICO
601,94dc337a82ede3ede4646e699de68af8416bab6c6312b2...,QUÍMICA BIOLÓGICA,CIÊNCIAS BIOLÓGICAS,1988,8,CIÊNCIAS BIOLÓGICAS II,ACADÊMICO


In [908]:
#Salvando o dataframe final em um csv
programas_final.to_csv(f'{processed_dir}/programas.csv', index=False)

### Ano_programas

In [909]:
programas

,AN_BASE,CD_PROGRAMA_IES,NM_PROGRAMA_IES,NM_GRANDE_AREA_CONHECIMENTO,NM_GRAU_PROGRAMA,CD_CONCEITO_PROGRAMA,AN_INICIO_PROGRAMA,AN_INICIO_CURSO,IN_REDE,DS_SITUACAO_PROGRAMA,CD_AREA_AVALIACAO,NM_AREA_AVALIACAO,NM_MODALIDADE_PROGRAMA,ID_PROGRAMA_HASH
0,2017,31001017005P0,ESTATÍSTICA,CIÊNCIAS EXATAS E DA TERRA,MESTRADO/DOUTORADO,5,1981,1981/2001,False,EM FUNCIONAMENTO,1,MATEMÁTICA / PROBABILIDADE E ESTATÍSTICA,ACADÊMICO,24e4155389ae312fc18cc8f84e9470892dd97aacbf6b84...
1,2017,31001017162P8,SAÚDE PERINATAL,CIÊNCIAS DA SAÚDE,MESTRADO PROFISSIONAL,3,2015,2015,False,EM FUNCIONAMENTO,16,MEDICINA II,PROFISSIONAL,b1437042c91b7c1df69f90bce30db83a9a74038c187595...
2,2017,31001017158P0,ENGENHARIA DA NANOTECNOLOGIA,ENGENHARIAS,MESTRADO/DOUTORADO,4,2014,2014/2014,False,EM FUNCIONAMENTO,12,ENGENHARIAS II,ACADÊMICO,4c7293f22405bb76daab3eb47a05d40a02edb1b16b54a7...
3,2017,31001017156P8,ENSINO DE QUÍMICA,MULTIDISCIPLINAR,MESTRADO PROFISSIONAL,3,2014,2014,False,EM FUNCIONAMENTO,46,ENSINO,PROFISSIONAL,a5e8ff9afe24a995222b925db858d1a020925a15b36af1...
4,2017,31001017151P6,NUTRIÇÃO CLÍNICA,CIÊNCIAS DA SAÚDE,MESTRADO PROFISSIONAL,3,2013,2013,False,EM FUNCIONAMENTO,50,NUTRIÇÃO,PROFISSIONAL,57063ae0b5389c020479940f5f35ce792b0d9f7f464bb2...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1294,2014,31001017154P5,CIÊNCIA E TECNOLOGIA FARMACÊUTICA,CIÊNCIAS DA SAÚDE,MESTRADO PROFISSIONAL,3,2013,2013,False,EM FUNCIONAMENTO,19,FARMÁCIA,PROFISSIONAL,8d1026cd82f31ff928a7b7c9051c8284b668423f959e04...
1295,2014,31001017155P1,ENSINO DE HISTÓRIA,CIÊNCIAS HUMANAS,MESTRADO PROFISSIONAL,4,2014,2014,True,EM FUNCIONAMENTO,40,HISTÓRIA,PROFISSIONAL,99927fd51fa27594138bcccbaa247b9f230e1cc0d92304...
1296,2014,31001017156P8,ENSINO DE QUÍMICA,MULTIDISCIPLINAR,MESTRADO PROFISSIONAL,3,2014,2014,False,EM FUNCIONAMENTO,46,ENSINO,PROFISSIONAL,a5e8ff9afe24a995222b925db858d1a020925a15b36af1...
1297,2014,31001017157P4,ARTES DA CENA,"LINGÜÍSTICA, LETRAS E ARTES",MESTRADO,4,2014,2014,False,EM FUNCIONAMENTO,11,ARTES / MÚSICA,ACADÊMICO,ad401b13681ece1c71974b2dfc2708f4f649864fae9457...


In [910]:
#Basicamente, aqui iremos apenas definir as colunas que queremos acompanhar a evolução e extraí-las para obter o dataframe final
ano_programas_cols = ['AN_BASE','ID_PROGRAMA_HASH', 'CD_CONCEITO_PROGRAMA', 'IN_REDE', 'DS_SITUACAO_PROGRAMA']
ano_programas = programas[ano_programas_cols]

In [911]:
#Olhando a tabela completa
ano_programas

,AN_BASE,ID_PROGRAMA_HASH,CD_CONCEITO_PROGRAMA,IN_REDE,DS_SITUACAO_PROGRAMA
0,2017,24e4155389ae312fc18cc8f84e9470892dd97aacbf6b84...,5,False,EM FUNCIONAMENTO
1,2017,b1437042c91b7c1df69f90bce30db83a9a74038c187595...,3,False,EM FUNCIONAMENTO
2,2017,4c7293f22405bb76daab3eb47a05d40a02edb1b16b54a7...,4,False,EM FUNCIONAMENTO
3,2017,a5e8ff9afe24a995222b925db858d1a020925a15b36af1...,3,False,EM FUNCIONAMENTO
4,2017,57063ae0b5389c020479940f5f35ce792b0d9f7f464bb2...,3,False,EM FUNCIONAMENTO
...,...,...,...,...,...
1294,2014,8d1026cd82f31ff928a7b7c9051c8284b668423f959e04...,3,False,EM FUNCIONAMENTO
1295,2014,99927fd51fa27594138bcccbaa247b9f230e1cc0d92304...,4,True,EM FUNCIONAMENTO
1296,2014,a5e8ff9afe24a995222b925db858d1a020925a15b36af1...,3,False,EM FUNCIONAMENTO
1297,2014,ad401b13681ece1c71974b2dfc2708f4f649864fae9457...,4,False,EM FUNCIONAMENTO


In [912]:
#Salvando dataframe ano_programas para arquivo
ano_programas.to_csv(f'{processed_dir}/ano_programas.csv', index=False)

### Cursos

In [913]:
#Agora, será necessário extrair a tabela 'cursos' de dentro da tabela 'programas'
#Lembrando que alguns programas oferecem mestrado e doutorado simultaneamente
# Logo, precisaremos de alguma lógica para lidar com isso
print(f'Valores únicos: {programas['NM_GRAU_PROGRAMA'].unique()}')
print()
print(programas[['NM_GRAU_PROGRAMA', 'AN_INICIO_CURSO']])

Valores únicos: ['MESTRADO/DOUTORADO' 'MESTRADO PROFISSIONAL' 'MESTRADO' 'DOUTORADO']

           NM_GRAU_PROGRAMA AN_INICIO_CURSO
0        MESTRADO/DOUTORADO       1981/2001
1     MESTRADO PROFISSIONAL            2015
2        MESTRADO/DOUTORADO       2014/2014
3     MESTRADO PROFISSIONAL            2014
4     MESTRADO PROFISSIONAL            2013
...                     ...             ...
1294  MESTRADO PROFISSIONAL            2013
1295  MESTRADO PROFISSIONAL            2014
1296  MESTRADO PROFISSIONAL            2014
1297               MESTRADO            2014
1298     MESTRADO/DOUTORADO       2014/2014

[1299 rows x 2 columns]


In [914]:
def gerador_tabela_cursos(
    df_programas: pd.DataFrame,
    coluna_id_programa: str = 'ID_PROGRAMA_HASH', 
    coluna_grau: str = 'NM_GRAU_PROGRAMA',
    coluna_ano: str = 'AN_INICIO_CURSO'
) -> pd.DataFrame:
    """
    Desagrega programas que possuem graus combinados (ex: 'MESTRADO/DOUTORADO')
    e anos de início correspondentes (ex: '1981/2001') em linhas separadas.

    Após a desagregação, a função retorna um novo DataFrame contendo apenas
    as colunas CD_PROGRAMA_IES (ou o nome fornecido), NM_GRAU_PROGRAMA (ou o nome fornecido),
    e AN_INICIO_CURSO (ou o nome fornecido), com entradas duplicadas removidas.

    Args:
        df_programas (pd.DataFrame): O DataFrame de entrada.
        coluna_id_programa (str): O nome da coluna que contém o código do programa (ex: 'ID_PROGRAMA_HASH').
        coluna_grau (str): O nome da coluna que contém o grau do programa (padrão: 'NM_GRAU_PROGRAMA').
        coluna_ano (str): O nome da coluna que contém o ano de início do curso (padrão: 'AN_INICIO_CURSO').

    Returns:
        pd.DataFrame: Um novo DataFrame com as colunas especificadas, graus desagregados
                      e entradas duplicadas removidas.

    Raises:
        ValueError: Se as colunas especificadas não existirem no DataFrame.
    """
    # Garante que as colunas existem no DataFrame de entrada
    colunas_necessarias = [coluna_id_programa, coluna_grau, coluna_ano]
    for col in colunas_necessarias:
        if col not in df_programas.columns:
            raise ValueError(f"A coluna '{col}' não foi encontrada no DataFrame. Verifique o nome da coluna.")

    # Trabalha em uma cópia para não modificar o DataFrame original
    df_trabalho = df_programas.copy()

    # Identifica as linhas que contêm '/' na coluna de grau especificada
    mascara_combinados = df_trabalho[coluna_grau].str.contains('/', na=False)
    programas_combinados = df_trabalho[mascara_combinados].copy()
    programas_outros_graus = df_trabalho[~mascara_combinados].copy()

    df_novas_linhas = pd.DataFrame() # DataFrame vazio para acumular as novas linhas geradas

    if not programas_combinados.empty:
        # Divide tanto a coluna de graus quanto a de anos, usando os nomes passados
        graus_divididos = programas_combinados[coluna_grau].str.split('/', expand=True)
        anos_divididos = programas_combinados[coluna_ano].str.split('/', expand=True)

        # Itera sobre as possíveis "partes"
        for i in range(max(len(graus_divididos.columns), len(anos_divididos.columns))):
            nome_grau_parte = graus_divididos.get(i)
            ano_curso_parte = anos_divididos.get(i)

            if nome_grau_parte is not None and ano_curso_parte is not None:
                df_parte = programas_combinados.copy()

                df_parte[coluna_grau] = nome_grau_parte
                df_parte[coluna_ano] = ano_curso_parte

                df_novas_linhas = pd.concat([df_novas_linhas, df_parte], ignore_index=True)

        df_novas_linhas = df_novas_linhas.dropna(subset=[coluna_ano])

    # Concatena o DataFrame de programas sem combinação e as novas linhas geradas
    programas_final_completo = pd.concat([programas_outros_graus, df_novas_linhas], ignore_index=True)

    # --- Nova Lógica para selecionar colunas e remover duplicatas ---

    # Seleciona apenas as colunas desejadas
    df_resultado = programas_final_completo[[coluna_id_programa, coluna_grau, coluna_ano]].copy()

    # Opcional: Converte a coluna de ano de início para tipo numérico
    df_resultado[coluna_ano] = pd.to_numeric(
        df_resultado[coluna_ano], errors='coerce'
    ).astype('Int64') # Usando Int64 para aceitar nulos, como discutido anteriormente

    # Remove entradas duplicadas com base nas três colunas
    df_resultado.drop_duplicates(inplace=True)

    return df_resultado

In [915]:
cursos = gerador_tabela_cursos(programas) #Pode usar a tabela inicial, já que ela vai ter todas as informações

In [916]:
cursos

,ID_PROGRAMA_HASH,NM_GRAU_PROGRAMA,AN_INICIO_CURSO
0,b1437042c91b7c1df69f90bce30db83a9a74038c187595...,MESTRADO PROFISSIONAL,2015
1,a5e8ff9afe24a995222b925db858d1a020925a15b36af1...,MESTRADO PROFISSIONAL,2014
2,57063ae0b5389c020479940f5f35ce792b0d9f7f464bb2...,MESTRADO PROFISSIONAL,2013
3,ad401b13681ece1c71974b2dfc2708f4f649864fae9457...,MESTRADO,2014
4,99927fd51fa27594138bcccbaa247b9f230e1cc0d92304...,MESTRADO PROFISSIONAL,2014
...,...,...,...
1387,d86eb1634319da0ad6d28eed7359f409f6dad398035c4a...,DOUTORADO,2009
1516,4d73420d1163e1a371bdab470bccdab8d6370ee906d832...,DOUTORADO,2019
1583,ad401b13681ece1c71974b2dfc2708f4f649864fae9457...,DOUTORADO,2020
1605,5d737a8b9c2395b45be2686df2c234f68097b6578aa198...,DOUTORADO,2021


In [917]:
#Salvando a tabela 'cursos' para um csv
cursos.to_csv(f'{processed_dir}/cursos.csv', index=False)

### Produção

In [918]:
#Importando df filtrada
producao = pd.read_csv(f'{filtered_dir}/producao.csv')
producao

,AN_BASE,CD_PROGRAMA_IES,NM_PROGRAMA_IES,ID_ADD_PRODUCAO_INTELECTUAL,ID_PESSOA_DOCENTE,ID_PESSOA_DISCENTE,TP_AUTOR,NM_TP_CATEGORIA_DOCENTE,NM_NIVEL_DISCENTE,SG_ENTIDADE_ENSINO
0,2022,31001017015P5,CIÊNCIAS BIOLÓGICAS (FARMACOLOGIA E QUÍMICA ME...,35295536,535426.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
1,2022,31001017015P5,CIÊNCIAS BIOLÓGICAS (FARMACOLOGIA E QUÍMICA ME...,35295538,141762.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
2,2022,31001017015P5,CIÊNCIAS BIOLÓGICAS (FARMACOLOGIA E QUÍMICA ME...,35295539,476439.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
3,2022,31001017015P5,CIÊNCIAS BIOLÓGICAS (FARMACOLOGIA E QUÍMICA ME...,35295539,15859.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
4,2022,31001017015P5,CIÊNCIAS BIOLÓGICAS (FARMACOLOGIA E QUÍMICA ME...,35295540,539116.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
...,...,...,...,...,...,...,...,...,...,...
145021,2017,31001017103P1,URBANISMO,30988886,15187.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
145022,2017,31001017103P1,URBANISMO,30988907,385663.0,NaN,DOCENTE,PERMANENTE,NaN,UFRJ
145023,2017,31001017103P1,URBANISMO,30988907,15522.0,NaN,DOCENTE,COLABORADOR,NaN,UFRJ
145024,2017,31001017103P1,URBANISMO,30988907,NaN,767619.0,DISCENTE,NaN,MESTRADO,UFRJ


In [919]:
producao['NM_NIVEL_DISCENTE'].unique()

array([nan, 'MESTRADO', 'DOUTORADO', 'MESTRADO PROFISSIONAL'],
      dtype=object)

In [920]:
#Fundindo colunas ID_PESSOA_DISCENTE e ID_PESSOA_DOCENTE na coluna ID_PESSOA
producao['ID_PESSOA'] = producao['ID_PESSOA_DOCENTE'].fillna(
    producao['ID_PESSOA_DISCENTE'] #Substituinto NAs de ID_PESSOA_DOCENTE pelos valores em ID_PESSOA_DISCENTE
    ).astype(
        'int' #'ID_PESSOA_DOCENTE' convertido para int para evitar a permanência de floats graças aos nan anteriores
    ).rename(
        'ID_PESSOA' #Renomeando ID_PESSOA_DOCENTE para ID_PESSOA
    )  

In [921]:
producao[['ID_PESSOA', 'ID_PESSOA_DISCENTE', 'ID_PESSOA_DOCENTE', 'TP_AUTOR']]

,ID_PESSOA,ID_PESSOA_DISCENTE,ID_PESSOA_DOCENTE,TP_AUTOR
0,535426,NaN,535426.0,DOCENTE
1,141762,NaN,141762.0,DOCENTE
2,476439,NaN,476439.0,DOCENTE
3,15859,NaN,15859.0,DOCENTE
4,539116,NaN,539116.0,DOCENTE
...,...,...,...,...
145021,15187,NaN,15187.0,DOCENTE
145022,385663,NaN,385663.0,DOCENTE
145023,15522,NaN,15522.0,DOCENTE
145024,767619,767619.0,NaN,DISCENTE


In [922]:
#Gerando colunas com os hashes baseados nos ids originais
producao['ID_PESSOA_HASH'] = converter_ids_para_hashes(producao['ID_PESSOA'])
producao['ID_PRODUCAO_HASH'] = converter_ids_para_hashes(producao['ID_ADD_PRODUCAO_INTELECTUAL'])
producao['ID_PROGRAMA_HASH'] = converter_ids_para_hashes(producao['CD_PROGRAMA_IES'])
producao[['ID_PESSOA', 'ID_PESSOA_HASH', 'ID_ADD_PRODUCAO_INTELECTUAL', 'ID_PRODUCAO_HASH', 'CD_PROGRAMA_IES', 'ID_PROGRAMA_HASH']]

,ID_PESSOA,ID_PESSOA_HASH,ID_ADD_PRODUCAO_INTELECTUAL,ID_PRODUCAO_HASH,CD_PROGRAMA_IES,ID_PROGRAMA_HASH
0,535426,b87390e764d953e77281bd7eb52a0bc0794d47aae22607...,35295536,a3a7aded76b2cce792f0b0af5fc7a1ddbcd4041d8fcbb2...,31001017015P5,55f257f8aa680677a92764b1a4313a1e73ffba804fdcc5...
1,141762,177f148310a17c3904fc4cfe8664e60efab51c56cfb90a...,35295538,8e4cbf8cac6f52f45ea958ab56ac0652f81a3249f975fd...,31001017015P5,55f257f8aa680677a92764b1a4313a1e73ffba804fdcc5...
2,476439,fb4143f2035593a13237d445fd9b660049299a143dcec7...,35295539,03f0e8f8827e54485cc0f5679db84c0562f08a502a99a6...,31001017015P5,55f257f8aa680677a92764b1a4313a1e73ffba804fdcc5...
3,15859,c33d33113ca17c36bbda625be3f880fb213d92e700efcb...,35295539,03f0e8f8827e54485cc0f5679db84c0562f08a502a99a6...,31001017015P5,55f257f8aa680677a92764b1a4313a1e73ffba804fdcc5...
4,539116,35ec09a3b6d7ec3494bd3a78479b706c4d84c746cb0d8a...,35295540,e27526cac9cfa93a742624209dc69542b8380171c77615...,31001017015P5,55f257f8aa680677a92764b1a4313a1e73ffba804fdcc5...
...,...,...,...,...,...,...
145021,15187,9867963be32ed8a6bb682967f2d1ace9b228486896e475...,30988886,c68db59661efe5a6733d952e8e7219ec702c78d958da40...,31001017103P1,3f132af57a57b07c227c51c817b65c52aecaaf62cb51ed...
145022,385663,cca67d42c7745a5afb3d15db70ee4d66b6662aa7f9bda0...,30988907,14db438cb94f8658ef125886cebac15aab2a3aa23db7ed...,31001017103P1,3f132af57a57b07c227c51c817b65c52aecaaf62cb51ed...
145023,15522,c247246be31b52e2f6386f69f4ece4fc22343e43162672...,30988907,14db438cb94f8658ef125886cebac15aab2a3aa23db7ed...,31001017103P1,3f132af57a57b07c227c51c817b65c52aecaaf62cb51ed...
145024,767619,2e4b86c7c855952aba2d083f901837aeed9352828867a3...,30988907,14db438cb94f8658ef125886cebac15aab2a3aa23db7ed...,31001017103P1,3f132af57a57b07c227c51c817b65c52aecaaf62cb51ed...


In [923]:
#Checando entradas com autores que não estão na tabela 'pessoas'
print(f"{(~producao['ID_PESSOA_HASH'].isin(pessoas['ID_PESSOA_HASH'])).sum()} linhas com autores que não estão entre os docentes ou discentes da UFRJ") 
print(f"{(~producao.drop_duplicates(subset=['ID_PESSOA'])['ID_PESSOA_HASH'].isin(pessoas['ID_PESSOA_HASH'])).sum()} autores que não estão dentre discentes ou docentes da UFRJ")

474 linhas com autores que não estão entre os docentes ou discentes da UFRJ
69 autores que não estão dentre discentes ou docentes da UFRJ


In [924]:
#Removendo autores não presentes na tabela 'pessoas'
producao = producao[producao['ID_PESSOA_HASH'].isin(pessoas['ID_PESSOA_HASH'])]

In [925]:
#Mantendo apenas as colunas de interesse
producao_final = producao[['AN_BASE', 'ID_PROGRAMA_HASH', 'ID_PRODUCAO_HASH', 'ID_PESSOA_HASH', 'TP_AUTOR', 'NM_NIVEL_DISCENTE', 'NM_TP_CATEGORIA_DOCENTE']].convert_dtypes() #convert_dtypes tenta inferir o melhor tipo para cada coluna

In [926]:
producao_final

,AN_BASE,ID_PROGRAMA_HASH,ID_PRODUCAO_HASH,ID_PESSOA_HASH,TP_AUTOR,NM_NIVEL_DISCENTE,NM_TP_CATEGORIA_DOCENTE
0,2022,55f257f8aa680677a92764b1a4313a1e73ffba804fdcc5...,a3a7aded76b2cce792f0b0af5fc7a1ddbcd4041d8fcbb2...,b87390e764d953e77281bd7eb52a0bc0794d47aae22607...,DOCENTE,<NA>,PERMANENTE
1,2022,55f257f8aa680677a92764b1a4313a1e73ffba804fdcc5...,8e4cbf8cac6f52f45ea958ab56ac0652f81a3249f975fd...,177f148310a17c3904fc4cfe8664e60efab51c56cfb90a...,DOCENTE,<NA>,PERMANENTE
2,2022,55f257f8aa680677a92764b1a4313a1e73ffba804fdcc5...,03f0e8f8827e54485cc0f5679db84c0562f08a502a99a6...,fb4143f2035593a13237d445fd9b660049299a143dcec7...,DOCENTE,<NA>,PERMANENTE
3,2022,55f257f8aa680677a92764b1a4313a1e73ffba804fdcc5...,03f0e8f8827e54485cc0f5679db84c0562f08a502a99a6...,c33d33113ca17c36bbda625be3f880fb213d92e700efcb...,DOCENTE,<NA>,PERMANENTE
4,2022,55f257f8aa680677a92764b1a4313a1e73ffba804fdcc5...,e27526cac9cfa93a742624209dc69542b8380171c77615...,35ec09a3b6d7ec3494bd3a78479b706c4d84c746cb0d8a...,DOCENTE,<NA>,PERMANENTE
...,...,...,...,...,...,...,...
145021,2017,3f132af57a57b07c227c51c817b65c52aecaaf62cb51ed...,c68db59661efe5a6733d952e8e7219ec702c78d958da40...,9867963be32ed8a6bb682967f2d1ace9b228486896e475...,DOCENTE,<NA>,PERMANENTE
145022,2017,3f132af57a57b07c227c51c817b65c52aecaaf62cb51ed...,14db438cb94f8658ef125886cebac15aab2a3aa23db7ed...,cca67d42c7745a5afb3d15db70ee4d66b6662aa7f9bda0...,DOCENTE,<NA>,PERMANENTE
145023,2017,3f132af57a57b07c227c51c817b65c52aecaaf62cb51ed...,14db438cb94f8658ef125886cebac15aab2a3aa23db7ed...,c247246be31b52e2f6386f69f4ece4fc22343e43162672...,DOCENTE,<NA>,COLABORADOR
145024,2017,3f132af57a57b07c227c51c817b65c52aecaaf62cb51ed...,14db438cb94f8658ef125886cebac15aab2a3aa23db7ed...,2e4b86c7c855952aba2d083f901837aeed9352828867a3...,DISCENTE,MESTRADO,<NA>


In [927]:
#Salvando dataframe producao_final como csv
producao_final.to_csv(f'{processed_dir}/producao.csv', index=False)

Com os dados processados, agora é possível usá-los para popular o banco de dados do painel de dados de pesquisa e pós-graduação da UFRJ.